# Huấn luyện DemucsLite V2 trên Colab

Bản này mặc định **chỉ train DemucsLite V2** và nghe checkpoint V2 vừa xuất. Giữ PyTorch/NumPy của runtime Colab hiện tại, gồm Python 3.13 trở lên; không cài Demucs, torchaudio hoặc ép NumPy xuống 1.x. Chạy 36 test cho Lite, không chạy hai test HTDemucs. Chọn GPU trước khi chạy.

Fine-tune HTDemucs và đánh giá toàn bộ 50 bài test được tắt. Để dùng HTDemucs, mở notebook `main_improved_colab.ipynb` trong môi trường legacy riêng; notebook đó hướng dẫn chọn runtime Python 3.12. Chỉ đánh giá test sau khi đã khóa cấu hình.

Checkpoint tốt nhất nghĩa là tốt nhất trong lần chạy theo **validation SNR**; chưa bảo đảm vượt baseline hoặc đạt chất lượng tối ưu tuyệt đối. Không chỉnh hyperparameter dựa trên test.

Dataset Drive của bạn: `musdb18hq.zip`, 22.656 GB (khoảng 21.1 GiB). Giải nén WAV cần thêm dung lượng lớn; notebook kiểm tra dung lượng trước. GPU Colab không bị giữ vĩnh viễn bằng việc chống sleep trên máy cá nhân.


In [ ]:
import sys, os, subprocess, importlib.util, importlib.metadata
# Dùng PyTorch CUDA có sẵn trên Colab, không thay bằng wheel khác.
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Runtime này chưa có PyTorch. Chọn GPU runtime Colab có PyTorch sẵn.')
loaded_numpy = getattr(sys.modules.get('numpy'), '__version__', None)
requirements = {
    'numpy': ('numpy>=1.26', (1, 26)),
    'soundfile': ('soundfile>=0.12', (0, 12)),
    'tqdm': ('tqdm>=4.66', (4, 66)),
}
missing_or_old = []
for package, (specification, minimum) in requirements.items():
    try:
        installed = importlib.metadata.version(package)
        meets_minimum = tuple(int(part) for part in installed.split('.')[:2]) >= minimum
    except (importlib.metadata.PackageNotFoundError, ValueError):
        meets_minimum = False
    if not meets_minimum:
        missing_or_old.append(specification)
if missing_or_old:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_or_old])
if loaded_numpy is not None and loaded_numpy != importlib.metadata.version('numpy'):
    raise RuntimeError('NumPy đã được kernel import trước khi nâng phiên bản. Restart session rồi chạy lại từ đầu.')
import numpy as np
import torch
import soundfile as sf
# Kiểm tra bridge NumPy ↔ PyTorch thay vì chặn theo phiên bản Python.
probe = np.zeros((2, 16), dtype=np.float32)
assert np.array_equal(torch.from_numpy(probe).numpy(), probe)
print('Python:', sys.version, 'PyTorch:', torch.__version__, 'NumPy:', np.__version__,
      'SoundFile:', sf.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime. Không train dài trên CPU.')
print(torch.cuda.get_device_name(0))


## 1. Mount Drive và chọn nơi lưu

Drive mount có thể yêu cầu bạn đăng nhập/cấp quyền qua giao diện Google. Checkpoint đặt trong MyDrive; dữ liệu WAV sẽ giải nén vào `/content` để đọc nhanh hơn.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
PROJECT = Path('/content/demucs_improved')
PROJECT.mkdir(exist_ok=True)
SAVE_ROOT = Path('/content/drive/MyDrive/DemucsImproved')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
(PROJECT / 'audio_sep').mkdir(exist_ok=True)
(PROJECT / 'tests').mkdir(exist_ok=True)
print('Kết quả sẽ lưu vào:', SAVE_ROOT)


## 2. Code đã cải tiến

Các cell dưới chỉ ghi module. Có thể chỉnh code tại đây trước khi chạy; bản gốc không bị ghi đè.


In [ ]:
%%writefile audio_sep/__init__.py
"""Reproducible music source separation experiments."""


In [ ]:
%%writefile audio_sep/compat.py
"""Scoped compatibility for trusted official Demucs 4 checkpoints on PyTorch 2.6+."""
import os
from threading import Lock

_LOAD_LOCK = Lock()


def official_model(name):
    if name not in ("htdemucs", "htdemucs_ft"):
        raise ValueError("Only official htdemucs/htdemucs_ft names are supported.")
    from demucs.pretrained import get_model
    # Demucs 4 torch.load omits weights_only. PyTorch 2.6 changed its default.
    # Limit the compatibility setting to official, checksum-verified downloads.
    with _LOAD_LOCK:
        key = "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"
        previous = os.environ.get(key)
        os.environ[key] = "1"
        try:
            return get_model(name)
        finally:
            if previous is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = previous


In [ ]:
%%writefile audio_sep/data.py
"""Audited MUSDB18-HQ loading and stereo, source-consistent augmentation.

MUSDB18-HQ contains 100 physical training tracks and 50 test tracks.  The
official 14 validation tracks are selected by name, before drawing any crops.
Never split segments from one song across training and validation.

The exact PR - Oh No test track has a documented left-channel erratum and an
additional observed right-channel residual. Its residuals are warnings under
the explicit known-track policy; original benchmark mixture/stems are preserved.
https://sigsep.github.io/datasets/musdb.html#errata
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path
import warnings
from typing import Sequence

import numpy as np
import soundfile as sf
import torch
from torch.utils.data import Dataset

DEFAULT_SOURCES = ("vocals", "drums", "bass", "other")
# https://github.com/sigsep/sigsep-mus-db/blob/master/musdb/configs/mus.yaml
VALIDATION_TRACKS = frozenset(
    (
        "Actions - One Minute Smile",
        "Clara Berry And Wooldog - Waltz For My Victims",
        "Johnny Lokke - Promises & Lies",
        "Patrick Talbot - A Reason To Leave",
        "Triviul - Angelsaint",
        "Alexander Ross - Goodbye Bolero",
        "Fergessen - Nos Palpitants",
        "Leaf - Summerghost",
        "Skelpolu - Human Mistakes",
        "Young Griffo - Pennies",
        "ANiMAL - Rockshow",
        "James May - On The Line",
        "Meaxic - Take A Step",
        "Traffic Experiment - Sirens",
    )
)
_ALL_FILES = ("mixture", *DEFAULT_SOURCES)
KNOWN_LEFT_MIX_ERRATUM = "musdb18hq_pr_oh_no_left_mixture"
KNOWN_MIX_RESIDUAL_POLICY = "warn_known_artistic_mix"
MUSDB_ERRATA_URL = "https://sigsep.github.io/datasets/musdb.html#errata"


def _write_json(path: str | Path, value: object) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def save_manifest(records: Sequence[dict], path: str | Path) -> None:
    """Save metadata; audio is neither copied nor downloaded."""
    _write_json(path, {"schema_version": 1, "sources": list(DEFAULT_SOURCES), "tracks": list(records)})


def load_manifest(path: str | Path, root: str | Path | None = None) -> list[dict]:
    """Read a manifest, optionally relocating its paths under a new dataset root."""
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    records = payload if isinstance(payload, list) else payload["tracks"]
    records = [dict(record) for record in records]
    if root is not None:
        root = Path(root).expanduser().resolve()
        for record in records:
            relative = Path(record["relative_path"])
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError(f"Unsafe relative_path for {record['track_name']}: {relative}")
            record["path"] = str(root / relative)
    return records


def _quantization_step(subtype: str) -> float:
    return {"PCM_U8": 2.0**-7, "PCM_S8": 2.0**-7, "PCM_16": 2.0**-15,
            "PCM_24": 2.0**-23, "PCM_32": 2.0**-31}.get(subtype, 0.0)


def _rms(audio: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(audio.astype(np.float64)))))


def scan_dataset(
    root: str | Path,
    output: str | Path | None = None,
    strict: bool = True,
    *,
    audit_output: str | Path | None = None,
    probe_seconds: float = 1.0,
    allow_known_errata: bool = True,
) -> list[dict]:
    """Audit and index a local MUSDB18-HQ WAV dataset.

    Required layout: ``root/train/<song>/{mixture,vocals,drums,bass,other}.wav``
    and ``root/test/<song>/...``. A wrapping directory under ``root`` is allowed.
    ``strict=False`` relaxes dataset counts only, for small development fixtures.
    Every WAV must still be 44.1 kHz stereo with an identical, positive length.

    Three spread probes per song check finite samples, stem activity, and the
    residual between mixture and sum of stems. WAV rounding is tolerated: the
    warning floor is five quantization steps; a much larger residual raises an
    error because it suggests an alignment or export problem. These probes do
    not certify every sample; the loader checks every crop it reads as well.

    By default, only physical ``test/PR - Oh No`` permits residuals on both
    channels: the left is documented by SigSep, the right is additionally
    observed in the supplied dataset and is not attributed to that errata.
    Shape, finite samples and every other song remain checked as before.
    ``allow_known_errata=False`` rejects residuals on either channel. The scanner
    does not modify audio, replace the original mixture or exclude the track.
    """
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {root}")
    if probe_seconds <= 0:
        raise ValueError("probe_seconds must be positive")
    candidates = sorted(root.rglob("mixture.wav"))
    if not candidates:
        raise ValueError(f"No mixture.wav found in {root}; extract MUSDB18-HQ first.")
    records: list[dict] = []
    audits: list[dict] = []
    errors: list[str] = []
    messages: list[str] = []
    physical_names: dict[str, set[str]] = {"train": set(), "test": set()}
    for mixture_path in candidates:
        track_path = mixture_path.parent
        track_name = track_path.name
        split = track_path.parent.name
        label = f"{split}/{track_name}"
        known_issue = KNOWN_LEFT_MIX_ERRATUM if split == "test" and track_name == "PR - Oh No" else None
        residual_policy = (KNOWN_MIX_RESIDUAL_POLICY if allow_known_errata else "reject_residuals") if known_issue else None
        if split not in physical_names:
            errors.append(f"{label}: expected track directly under train/ or test/.")
            continue
        if track_name in physical_names[split]:
            errors.append(f"{label}: duplicate track name (possibly two dataset copies).")
            continue
        physical_names[split].add(track_name)
        infos: dict[str, sf.SoundFile] = {}
        track_errors: list[str] = []
        for name in _ALL_FILES:
            path = track_path / f"{name}.wav"
            if not path.is_file():
                track_errors.append(f"missing {name}.wav")
                continue
            try:
                infos[name] = sf.info(path)
            except (RuntimeError, sf.LibsndfileError) as exc:
                track_errors.append(f"cannot read {name}.wav: {exc}")
        if "mixture" in infos:
            expected = infos["mixture"]
            for name, info in infos.items():
                if info.samplerate != 44100 or info.channels != 2:
                    track_errors.append(f"{name}.wav: expected 44100 Hz stereo, got {info.samplerate} Hz/{info.channels} channels")
                if info.frames <= 0 or info.frames != expected.frames:
                    track_errors.append(f"{name}.wav: {info.frames} frames, mixture has {expected.frames}")
        if track_errors:
            errors.extend(f"{label}: {error}" for error in track_errors)
            audits.append({"track_name": track_name, "physical_subset": split,
                           "known_issue": known_issue, "residual_policy": residual_policy, "errors": track_errors})
            continue
        expected = infos["mixture"]
        length = min(expected.frames, max(1, round(probe_seconds * expected.samplerate)))
        offsets = sorted(set((0, (expected.frames - length) // 2, expected.frames - length)))
        probes = []
        quantization = max(_quantization_step(info.subtype) for info in infos.values())
        for offset in offsets:
            audio = {
                name: sf.read(track_path / f"{name}.wav", start=offset, frames=length,
                              always_2d=True, dtype="float32")[0]
                for name in _ALL_FILES
            }
            if any(value.shape != (length, 2) or not np.isfinite(value).all() for value in audio.values()):
                errors.append(f"{label}: non-finite samples or a truncated read at frame {offset}")
                continue
            summed = np.sum([audio[source].astype(np.float64) for source in DEFAULT_SOURCES], axis=0)
            residual = audio["mixture"] - summed
            residual_rms = _rms(residual)
            mixture_rms = _rms(audio["mixture"])
            residual_db = 20 * np.log10(max(residual_rms, 1e-12) / max(mixture_rms, 1e-12))
            channel_diagnostics = {}
            for index, channel in enumerate(("left", "right")):
                channel_residual = _rms(residual[:, index])
                channel_mixture = _rms(audio["mixture"][:, index])
                channel_diagnostics[channel] = {
                    "residual_rms": channel_residual,
                    "residual_relative_db": float(20 * np.log10(max(channel_residual, 1e-12)
                                                                 / max(channel_mixture, 1e-12))),
                    "residual_peak": float(np.max(np.abs(residual[:, index]))),
                    "warning_floor": max(5 * quantization, 1e-5, channel_mixture * 1e-3),
                    "failure_floor": max(20 * quantization, 0.01, channel_mixture * 0.05),
                }
            probe = {
                "start_frame": offset,
                "frames": length,
                "rms": {name: _rms(value) for name, value in audio.items()},
                "active_fraction": {
                    name: float(np.mean(np.max(np.abs(value), axis=1) > 1e-4))
                    for name, value in audio.items()
                },
                "residual_rms": residual_rms,
                "residual_relative_db": float(residual_db),
                "residual_peak": float(np.max(np.abs(residual))),
                "residual_by_channel": channel_diagnostics,
            }
            probes.append(probe)
            warning_floor = max(5 * quantization, 1e-5, mixture_rms * 1e-3)
            failure_floor = max(20 * quantization, 0.01, mixture_rms * 0.05)
            if known_issue:
                # Narrow exception for this physical test track only. Keep diagnostics
                # on both channels; a right mismatch is observed, not official errata.
                for channel, diagnostic in channel_diagnostics.items():
                    rms = diagnostic["residual_rms"]
                    db = diagnostic["residual_relative_db"]
                    if allow_known_errata and rms > diagnostic["warning_floor"]:
                        description = ("known MUSDB erratum on left channel; mixture/stem residual"
                                       if channel == "left" else "additional observed right-channel residual")
                        provenance = (MUSDB_ERRATA_URL if channel == "left"
                                      else "observed in supplied dataset; official erratum describes the left channel")
                        messages.append(f"{label}: {description} RMS {rms:.5g} ({db:.1f} dB) at frame {offset}; "
                                        f"policy {KNOWN_MIX_RESIDUAL_POLICY}; original mixture/stems retained; {provenance}")
                    elif rms > diagnostic["failure_floor"]:
                        errors.append(f"{label}: {channel} channel mixture/stem residual RMS {rms:.5g} "
                                      f"({db:.1f} dB) at frame {offset}; check alignment/export")
                    elif rms > diagnostic["warning_floor"]:
                        messages.append(f"{label}: {channel} channel mixture/stem residual RMS {rms:.5g} "
                                        f"({db:.1f} dB) at frame {offset}")
            elif residual_rms > failure_floor:
                errors.append(f"{label}: mixture/stem residual RMS {residual_rms:.5g} ({residual_db:.1f} dB) at frame {offset}; check alignment/export")
            elif residual_rms > warning_floor:
                messages.append(f"{label}: mixture/stem residual RMS {residual_rms:.5g} ({residual_db:.1f} dB) at frame {offset}")
        if probes and all(probe["rms"]["mixture"] < 1e-5 for probe in probes):
            messages.append(f"{label}: all mixture probes are nearly silent; inspect the song.")
        subset = "val" if split == "train" and track_name in VALIDATION_TRACKS else split
        record = {
            "track_name": track_name, "subset": subset, "physical_subset": split,
            "path": str(track_path), "relative_path": str(track_path.relative_to(root)),
            "frames": expected.frames, "sample_rate": expected.samplerate, "channels": expected.channels,
            "max_probe_residual_rms_by_channel": {
                channel: max((probe["residual_by_channel"][channel]["residual_rms"] for probe in probes), default=None)
                for channel in ("left", "right")
            },
        }
        if known_issue:
            record["known_issue"] = known_issue
            record["residual_policy"] = residual_policy
        records.append(record)
        audits.append({"track_name": track_name, "subset": subset, "known_issue": known_issue,
                       "residual_policy": residual_policy, "probes": probes})
    overlap = physical_names["train"] & physical_names["test"]
    if overlap:
        errors.append(f"Training/test track-name overlap: {sorted(overlap)}")
    if strict:
        for split, count in (("train", 100), ("test", 50)):
            if len(physical_names[split]) != count:
                errors.append(f"Expected {count} physical {split} tracks, found {len(physical_names[split])}")
        missing_validation = VALIDATION_TRACKS - physical_names["train"]
        if missing_validation:
            errors.append(f"Missing official validation tracks: {sorted(missing_validation)}")
    counts = {subset: sum(record["subset"] == subset for record in records) for subset in ("train", "val", "test")}
    if audit_output is None and output is not None:
        output_path = Path(output)
        audit_output = output_path.with_name(output_path.stem + ".audit.json")
    audit = {"schema_version": 1, "root": str(root), "counts": counts,
             "probe_seconds": probe_seconds, "warnings": messages, "errors": errors, "tracks": audits,
             "known_errata_policy": {"allow_known_errata": bool(allow_known_errata),
                                     "issue": KNOWN_LEFT_MIX_ERRATUM, "source": MUSDB_ERRATA_URL,
                                     "official_erratum_channels": ["left"],
                                     "additional_observed_residual_channels": ["right"],
                                     "residual_policy": KNOWN_MIX_RESIDUAL_POLICY if allow_known_errata else "reject_residuals",
                                     "original_mixture_stems_preserved": True}}
    if audit_output is not None:
        _write_json(audit_output, audit)
    if errors:
        raise ValueError("Dataset audit failed:\n- " + "\n- ".join(errors))
    for message in messages:
        warnings.warn(message, stacklevel=2)
    if output is not None:
        save_manifest(records, output)
    return records


class MUSDBDataset(Dataset):
    """Random training crops or deterministic spread validation crops.

    Returns ``(mixture[2,T], targets[S,2,T])`` in ``sources`` order. Random
    source remixing is restricted to the supplied training records. Source gain,
    polarity and stereo-channel swapping always affect both channels together;
    the augmented mixture is built *after* augmentation as the exact stem sum.
    A common scale avoids clipping without normalizing stems independently.

    Call ``set_epoch`` before creating each epoch's DataLoader iterator and use
    ``persistent_workers=False`` so workers receive the new deterministic seed.
    """

    def __init__(
        self,
        records: Sequence[dict],
        sources: Sequence[str] = DEFAULT_SOURCES,
        segment_seconds: float = 6.0,
        sample_rate: int = 44100,
        training: bool = True,
        crops_per_track: int = 32,
        remix_prob: float = 0.5,
        gain_db: float = 3.0,
        seed: int = 42,
        validation_crops: int = 3,
    ):
        self.records = [dict(record) for record in records]
        self.sources = tuple(sources)
        self.sample_rate = int(sample_rate)
        self.segment_frames = round(segment_seconds * self.sample_rate)
        self.training = bool(training)
        self.crops_per_track = int(crops_per_track if training else validation_crops)
        self.remix_prob = float(remix_prob)
        self.gain_db = float(gain_db)
        self.seed = int(seed)
        self.epoch = 0
        if not self.records:
            raise ValueError("No dataset records supplied")
        if self.sample_rate != 44100 or self.segment_frames <= 0:
            raise ValueError("MUSDB18-HQ requires sample_rate=44100 and a positive segment_seconds")
        if self.crops_per_track <= 0 or not 0 <= self.remix_prob <= 1 or self.gain_db < 0:
            raise ValueError("Invalid crops_per_track, validation_crops, remix_prob, or gain_db")
        if len(set(self.sources)) != len(self.sources) or set(self.sources) != set(DEFAULT_SOURCES):
            raise ValueError("sources must contain vocals, drums, bass, other exactly once")
        seen = set()
        for record in self.records:
            if training and record["subset"] != "train":
                raise ValueError(f"Training dataset includes {record['subset']} song: {record['track_name']}")
            identity = record["track_name"]
            if identity in seen:
                raise ValueError(f"Duplicate track: {identity}")
            seen.add(identity)
            if record["sample_rate"] != sample_rate or record["channels"] != 2 or record["frames"] <= 0:
                raise ValueError(f"Invalid manifest metadata for {identity}")
            for name in _ALL_FILES:
                info = sf.info(Path(record["path"]) / f"{name}.wav")
                if (info.samplerate, info.channels, info.frames) != (sample_rate, 2, record["frames"]):
                    raise ValueError(f"WAV/manifest mismatch for {identity}/{name}.wav; rescan the dataset")

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __len__(self) -> int:
        return len(self.records) * self.crops_per_track

    def _read(self, record: dict, source: str, start: int) -> np.ndarray:
        frames = min(self.segment_frames, record["frames"] - start)
        audio, sr = sf.read(Path(record["path"]) / f"{source}.wav", start=start, frames=frames,
                            dtype="float32", always_2d=True)
        if sr != self.sample_rate or audio.shape != (frames, 2) or not np.isfinite(audio).all():
            raise ValueError(f"Invalid audio crop: {record['track_name']}/{source}.wav at {start}")
        result = np.zeros((2, self.segment_frames), dtype=np.float32)
        result[:, :frames] = audio.T
        return result

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        if not 0 <= index < len(self):
            raise IndexError(index)
        record = self.records[index // self.crops_per_track]
        last_start = max(0, record["frames"] - self.segment_frames)
        if self.training:
            rng = np.random.default_rng(np.random.SeedSequence((self.seed, self.epoch, index)))
            start = int(rng.integers(last_start + 1))
            remix = rng.random() < self.remix_prob
            targets = []
            for source in self.sources:
                source_record = self.records[int(rng.integers(len(self.records)))] if remix else record
                source_start = int(rng.integers(max(0, source_record["frames"] - self.segment_frames) + 1)) if remix else start
                stem = self._read(source_record, source, source_start)
                stem *= np.float32(10 ** (rng.uniform(-self.gain_db, self.gain_db) / 20))
                if rng.random() < 0.5:
                    stem *= -1
                if rng.random() < 0.5:
                    stem = stem[::-1].copy()
                targets.append(stem)
            targets = np.stack(targets)
            mixture = targets.sum(axis=0)
        else:
            crop_index = index % self.crops_per_track
            fraction = 0.5 if self.crops_per_track == 1 else crop_index / (self.crops_per_track - 1)
            start = round(last_start * fraction)
            targets = np.stack([self._read(record, source, start) for source in self.sources])
            mixture = self._read(record, "mixture", start)
        peak = max(float(np.max(np.abs(mixture))), float(np.max(np.abs(targets))))
        if peak > 1.0:
            scale = np.float32(0.99 / peak)
            targets *= scale
            if self.training:
                # Re-summing preserves exact mixture consistency after rounding.
                mixture = targets.sum(axis=0)
            else:
                mixture *= scale
        target_tensor = torch.from_numpy(targets.copy())
        mixture_tensor = target_tensor.sum(dim=0) if self.training else torch.from_numpy(mixture.copy())
        return mixture_tensor, target_tensor


def main() -> None:
    parser = argparse.ArgumentParser(description="Audit local MUSDB18-HQ WAVs and write a manifest")
    parser.add_argument("root", type=Path)
    parser.add_argument("--output", type=Path, default=Path("data/manifest.json"))
    parser.add_argument("--no-strict", action="store_true", help="Allow a partial dataset; keep WAV checks")
    parser.add_argument("--probe-seconds", type=float, default=1.0)
    parser.add_argument("--reject-known-errata", action="store_true",
                        help="Reject residual mismatch on either channel of test/PR - Oh No as well")
    args = parser.parse_args()
    records = scan_dataset(args.root, args.output, strict=not args.no_strict, probe_seconds=args.probe_seconds,
                           allow_known_errata=not args.reject_known_errata)
    counts = {subset: sum(record["subset"] == subset for record in records) for subset in ("train", "val", "test")}
    print(json.dumps({"manifest": str(args.output), "counts": counts}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile audio_sep/evaluate.py
"""Full-track comparison: median 1s window SNR/SI-SDR, then macro over tracks."""
import argparse
import json
from pathlib import Path

import torch
from torch import nn
from tqdm import tqdm

from .data import DEFAULT_SOURCES, load_manifest
from .inference import load_audio, load_checkpoint, overlap_add, separate_official
from .metrics import metrics_per_source


def legacy_model(checkpoint, device="cpu"):
    from .legacy_model import DemucsLite
    ckpt = torch.load(checkpoint, map_location="cpu", weights_only=False)
    cfg = ckpt.get("model_config", {"channels": 32, "depth": 5})
    backbone = DemucsLite(sources=4, audio_channels=2, **cfg)
    backbone.load_state_dict(ckpt.get("model_state_dict", ckpt))

    class LegacyWrapper(nn.Module):
        sources = DEFAULT_SOURCES
        samplerate = 44100

        def __init__(self):
            super().__init__()
            self.backbone = backbone

        def forward(self, mix):
            return self.backbone(mix)

    return LegacyWrapper().to(device).eval()


def window_scores(estimate, target, sr=44100):
    rows = []
    for start in range(0, target.shape[-1], sr):
        ref = target[..., start:start + sr]
        est = estimate[..., start:start + sr]
        if ref.shape[-1] < sr // 2:
            continue
        rows.append(metrics_per_source(est[None], ref[None]))
    if not rows:
        rows = [metrics_per_source(estimate[None], target[None])]
    result = {}
    active = torch.cat([r["active"] for r in rows])
    for key in ("snr_db", "si_sdr_db"):
        scores = torch.cat([r[key] for r in rows])
        result[key] = [float(scores[:, i][active[:, i]].median()) if active[:, i].any() else None
                       for i in range(len(DEFAULT_SOURCES))]
    result["active_ratio"] = active.float().mean(0).tolist()
    silent = torch.cat([r["silent_estimate_energy"] for r in rows])
    result["silent_estimate_rms"] = [float(silent[:, i][~active[:, i]].mean().sqrt())
                                     if (~active[:, i]).any() else None for i in range(4)]
    return result


def summarize_tracks(rows):
    """Keep the primary macro metric identical, and handle an empty filtered view."""
    if not rows:
        return None
    summary = {}
    for key in ("snr_db", "si_sdr_db"):
        scores = torch.tensor([[float("nan") if x is None else x for x in r[key]] for r in rows],
                              dtype=torch.float32)
        per_source = torch.nanmean(scores, 0)
        summary[key] = {source: float(per_source[i]) for i, source in enumerate(DEFAULT_SOURCES)}
        summary[f"macro_{key}"] = float(torch.nanmean(per_source))
    return summary


def evaluation_report(rows, settings):
    """Report all tracks first, with a separate diagnostic view of flagged mixes.

    Neither view changes the inference audio or targets. The filtered view is
    explicitly secondary, so a known dataset anomaly cannot silently improve
    the advertised all-track score.
    """
    flagged = [row for row in rows if row.get("known_issue")]
    unflagged = [row for row in rows if not row.get("known_issue")]
    return {
        "settings": settings,
        "sources": list(DEFAULT_SOURCES),
        "mixture_policy": "original_mixture_wav",
        "metric_definition": "Median active 1-second window per track; arithmetic mean over tracks and sources. SNR is scale-dependent energy ratio, not BSS Eval SDR.",
        "summary": summarize_tracks(rows),
        "summary_without_flagged_tracks": summarize_tracks(unflagged),
        "track_counts": {"all": len(rows), "without_flagged": len(unflagged), "flagged": len(flagged)},
        "flagged_tracks": [
            {key: row[key] for key in ("track_name", "known_issue", "residual_policy") if key in row}
            for row in flagged
        ],
        "tracks": rows,
    }


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--manifest", required=True)
    choice = p.add_mutually_exclusive_group(required=True)
    choice.add_argument("--checkpoint")
    choice.add_argument("--legacy-checkpoint")
    choice.add_argument("--official", choices=["htdemucs", "htdemucs_ft"])
    p.add_argument("--subset", choices=["val", "test"], default="test")
    p.add_argument("--output", required=True)
    p.add_argument("--limit", type=int, default=0)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--shifts", type=int, default=2)
    args = p.parse_args()
    records = [r for r in load_manifest(args.manifest) if r["subset"] == args.subset]
    if args.limit:
        records = records[:args.limit]
    if not records:
        raise ValueError("No evaluation tracks.")
    if args.official:
        from .compat import official_model
        model = official_model(args.official).eval()
    elif args.legacy_checkpoint:
        model = legacy_model(args.legacy_checkpoint, args.device)
    else:
        model, _ = load_checkpoint(args.checkpoint, args.device)
    rows = []
    path = Path(args.output)
    path.parent.mkdir(parents=True, exist_ok=True)
    for record in tqdm(records, desc=f"Full-track {args.subset}"):
        directory = Path(record["path"])
        mixture = load_audio(directory / "mixture.wav")
        if args.official:
            estimate = separate_official(mixture, args.official, args.device,
                                         args.shifts, model=model)
        else:
            estimate = overlap_add(model, mixture, segment_seconds=10 if args.legacy_checkpoint else 6,
                                   device=args.device)
            estimate = estimate[[model.sources.index(s) for s in DEFAULT_SOURCES]]
        target = torch.stack([load_audio(directory / f"{source}.wav") for source in DEFAULT_SOURCES])
        if estimate.shape != target.shape:
            raise ValueError("Sample alignment changed during inference.")
        provenance = {key: record[key] for key in
                      ("known_issue", "residual_policy", "max_probe_residual_rms_by_channel")
                      if key in record}
        rows.append({"track_name": record["track_name"], **provenance, **window_scores(estimate, target)})
        # Partial readback survives a disconnected Colab session.
        path.write_text(json.dumps({"settings": vars(args), "sources": list(DEFAULT_SOURCES),
                                    "mixture_policy": "original_mixture_wav", "tracks": rows}, indent=2))
    report = evaluation_report(rows, vars(args))
    path.write_text(json.dumps(report, indent=2))
    print(json.dumps(report["summary"], indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile audio_sep/inference.py
"""Bounded-memory inference and lossless float WAV export."""
import argparse
import json
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from torch.nn import functional as F

from .data import DEFAULT_SOURCES


def load_audio(path, sample_rate=44100):
    data, sr = sf.read(str(path), dtype="float32", always_2d=True)
    if not np.isfinite(data).all() or not len(data):
        raise ValueError("Audio must be nonempty and finite.")
    if data.shape[1] == 1:
        data = np.repeat(data, 2, axis=1)
    if data.shape[1] != 2:
        raise ValueError("Provide mono/stereo audio; explicitly downmix multichannel input.")
    wav = torch.from_numpy(np.ascontiguousarray(data.T))
    if sr != sample_rate:
        from torchaudio.functional import resample
        wav = resample(wav, sr, sample_rate)
    return wav


def load_checkpoint(path, device="cpu", use_ema=None):
    # Only load checkpoints created by this project or other trusted sources.
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if ckpt.get("format_version") != 2:
        raise ValueError("Expected V2 checkpoint. Original weights require the original architecture.")
    if ckpt["model_kind"] == "lite":
        from .model import DemucsLiteV2
        model = DemucsLiteV2(**ckpt["model_config"])
    elif ckpt["model_kind"] == "pretrained":
        from .pretrained import PretrainedSeparator
        model = PretrainedSeparator(**ckpt["model_config"])
    else:
        raise ValueError("Unknown checkpoint architecture.")
    if use_ema is None:
        use_ema = ckpt.get("selected_weights") == "ema"
    state = ckpt.get("ema_state_dict") if use_ema else None
    model.load_state_dict(state if state is not None else ckpt["model_state_dict"])
    model.to(device).eval()
    return model, ckpt


@torch.inference_mode()
def overlap_add(model, mixture, segment_seconds=6.0, overlap=0.5, device="cpu"):
    """CPU full-song buffers; only one context window occupies accelerator memory."""
    if mixture.ndim != 2 or mixture.shape[-1] == 0:
        raise ValueError("Expected nonempty [channels,time] mixture.")
    if not 0 <= overlap < 1 or segment_seconds <= 0:
        raise ValueError("Positive segment and overlap in [0,1) required.")
    model.eval()
    sr = getattr(model, "samplerate", 44100)
    segment = max(1, round(segment_seconds * sr))
    if hasattr(model, "segment"):
        segment = min(segment, round(float(model.segment) * sr))
    hop = max(1, round(segment * (1 - overlap)))
    # Context on both boundaries avoids giving endpoints zero overlap weight.
    margin = segment // 2
    cpu_mix = mixture.detach().float().cpu()
    padded = F.pad(cpu_mix, (margin, margin))
    output = torch.zeros(len(model.sources), mixture.shape[0], padded.shape[-1])
    weights = torch.zeros(padded.shape[-1])
    window = torch.hann_window(segment, periodic=False).clamp_min(1e-3)
    for start in range(0, padded.shape[-1], hop):
        valid = min(segment, padded.shape[-1] - start)
        chunk = F.pad(padded[:, start:start + valid], (0, segment - valid))
        pred = model(chunk[None].to(device))[0, ..., :valid].float().cpu()
        output[..., start:start + valid] += pred * window[:valid]
        weights[start:start + valid] += window[:valid]
    output /= weights.clamp_min(1e-8)
    return output[..., margin:margin + cpu_mix.shape[-1]]


@torch.inference_mode()
def separate_official(mixture, name="htdemucs_ft", device="cpu", shifts=2, overlap=0.5, segment=7.8, model=None):
    from demucs.apply import apply_model
    from .compat import official_model
    # Keep ensemble on CPU: official apply_model moves one member at a time.
    model = official_model(name).eval() if model is None else model.eval()
    mixture = mixture.detach().float().cpu()
    if mixture.ndim != 2 or mixture.shape[0] != 2 or not mixture.shape[-1] or not torch.isfinite(mixture).all():
        raise ValueError("Expected finite nonempty stereo audio.")
    if shifts < 0 or not 0 <= overlap < 1 or segment <= 0:
        raise ValueError("Invalid shifts, overlap or segment.")
    reference = mixture.mean(dim=0)
    mean = reference.mean()
    std = reference.std(unbiased=False).clamp_min(1e-5)
    normalized = (mixture - mean) / std
    maximum = getattr(model, "max_allowed_segment", segment)
    segment = min(segment, float(maximum))
    result = apply_model(model, normalized[None], device=device, shifts=shifts,
                         split=True, overlap=overlap, segment=segment, progress=True)[0]
    result = result.float().cpu() * std + mean / len(model.sources)
    order = [model.sources.index(s) for s in DEFAULT_SOURCES]
    return result[order]


def save_stems(estimates, directory, sources=DEFAULT_SOURCES, sr=44100):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    if not torch.isfinite(estimates).all():
        raise ValueError("Nonfinite model estimates cannot be exported.")
    # FLOAT preserves amplitudes above 1 instead of silently clipping PCM16.
    for source, estimate in zip(sources, estimates):
        sf.write(directory / f"{source}.wav", estimate.cpu().numpy().T, sr, subtype="FLOAT")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("input")
    choice = p.add_mutually_exclusive_group(required=True)
    choice.add_argument("--official", choices=["htdemucs", "htdemucs_ft"])
    choice.add_argument("--checkpoint")
    p.add_argument("--output", required=True)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--segment", type=float)
    p.add_argument("--overlap", type=float, default=0.5)
    p.add_argument("--shifts", type=int, default=2)
    args = p.parse_args()
    wav = load_audio(args.input)
    if args.official:
        result = separate_official(wav, args.official, args.device, args.shifts,
                                   args.overlap, min(args.segment or 7.8, 7.8))
        sources = DEFAULT_SOURCES
    else:
        model, _ = load_checkpoint(args.checkpoint, args.device)
        result = overlap_add(model, wav, args.segment or 6, args.overlap, args.device)
        sources = model.sources
    save_stems(result, args.output, sources)
    Path(args.output, "inference.json").write_text(json.dumps(vars(args), indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile audio_sep/legacy_model.py
"""
Kiến trúc mạng U-Net cơ sở dựa trên Demucs (tinh gọn)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, stride)
        self.glu_conv = nn.Conv1d(out_channels, 2 * out_channels, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.glu_conv(x)
        x = F.glu(x, dim=1)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, is_last=False):
        super().__init__()
        self.glu_conv = nn.Conv1d(in_channels, 2 * in_channels, kernel_size=3, padding=1)
        self.deconv = nn.ConvTranspose1d(in_channels, out_channels, kernel_size, stride)
        self.is_last = is_last

    def forward(self, x, skip):
        x = x + skip
        x = self.glu_conv(x)
        x = F.glu(x, dim=1)
        x = self.deconv(x)
        if not self.is_last:
            x = F.relu(x)
        return x


class DemucsLite(nn.Module):
    def __init__(
        self,
        sources=4,
        audio_channels=2,
        channels=32,
        depth=5,
        growth=2,
        kernel_size=8,
        stride=4,
        lstm_layers=1,
    ):
        super().__init__()
        self.sources = sources
        self.audio_channels = audio_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.depth = depth

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        in_ch = audio_channels
        out_ch = channels
        channel_sizes = []
        for i in range(depth):
            self.encoders.append(EncoderBlock(in_ch, out_ch, kernel_size, stride))
            channel_sizes.append(out_ch)
            in_ch = out_ch
            out_ch = int(out_ch * growth)

        bottleneck_ch = channel_sizes[-1]
        self.lstm = nn.LSTM(
            input_size=bottleneck_ch,
            hidden_size=bottleneck_ch,
            num_layers=lstm_layers,
            bidirectional=True,
            batch_first=True,
        )
        self.lstm_proj = nn.Linear(2 * bottleneck_ch, bottleneck_ch)

        rev_channels = list(reversed(channel_sizes))
        for i in range(depth):
            in_ch = rev_channels[i]
            if i + 1 < depth:
                out_ch = rev_channels[i + 1]
            else:
                out_ch = sources * audio_channels
            is_last = i == depth - 1
            self.decoders.append(DecoderBlock(in_ch, out_ch, kernel_size, stride, is_last))

    def valid_length(self, length):
        for _ in range(self.depth):
            length = math.ceil((length - self.kernel_size) / self.stride) + 1
            length = max(length, 1)
        for _ in range(self.depth):
            length = (length - 1) * self.stride + self.kernel_size
        return int(length)

    def forward(self, mixture):
        original_length = mixture.shape[-1]
        target_length = self.valid_length(original_length)
        pad_amount = target_length - original_length
        x = F.pad(mixture, (0, pad_amount))

        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)

        x_lstm = x.permute(0, 2, 1)
        x_lstm, _ = self.lstm(x_lstm)
        x_lstm = self.lstm_proj(x_lstm)
        x = x_lstm.permute(0, 2, 1)

        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)

        x = x[..., :original_length]
        batch = x.shape[0]
        x = x.view(batch, self.sources, self.audio_channels, -1)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
%%writefile audio_sep/losses.py
"""Balanced waveform and stereo spectral objectives with explicit silence handling.

Unlike a global L1, each item/stem is normalized independently, preventing loud
drums from dominating quiet vocals. The mixture RMS floor limits amplification
of nearly silent labels. No residual-correlation heuristic is used: correlated
instruments are legitimate, and penalizing them is not proof of less leakage.
"""

from __future__ import annotations

from typing import Sequence

import torch
from torch import Tensor

from .metrics import _check_pair, si_sdr_values


def _source_scales(target: Tensor, mixture: Tensor, eps: float = 1e-4) -> Tensor:
    source_rms = target.square().mean(dim=(-2, -1)).clamp_min(eps**2).sqrt()
    mixture_rms = mixture.square().mean(dim=(-2, -1)).clamp_min(eps**2).sqrt()
    return torch.maximum(source_rms, 0.1 * mixture_rms[:, None]).clamp_min(eps)


def multi_resolution_stft_loss(estimate: Tensor, target: Tensor,
                               mixture: Tensor | None = None,
                               fft_sizes: Sequence[int] = (512, 1024, 2048, 4096)) -> Tensor:
    """Equal item/stem weighting; preserve L/R instead of averaging to mono.

    Spectral convergence is undefined for a silent target, so those entries use
    only log-magnitude error. Waveform/silence loss still penalizes hallucinations.
    Constant padding supports clips shorter than half an FFT window.
    """
    _check_pair(estimate, target)
    if not fft_sizes or any(size < 4 for size in fft_sizes):
        raise ValueError("fft_sizes must contain integers >= 4")
    with torch.autocast(device_type=estimate.device.type, enabled=False):
        estimate, target = estimate.float(), target.float()
        if mixture is None:
            mixture = target.sum(dim=1)
        mixture = mixture.float()
        scales = _source_scales(target, mixture)
        batch, sources, channels, samples = target.shape
        active = target.square().mean(dim=(-2, -1)) > 1e-6
        estimate = (estimate / scales[:, :, None, None]).reshape(-1, samples)
        target = (target / scales[:, :, None, None]).reshape(-1, samples)
        total = estimate.new_zeros(())
        for size in fft_sizes:
            window = torch.hann_window(size, device=estimate.device, dtype=torch.float32)
            est_mag = torch.stft(estimate, size, hop_length=size // 4, window=window,
                                 return_complex=True, center=True, pad_mode="constant").abs()
            tgt_mag = torch.stft(target, size, hop_length=size // 4, window=window,
                                 return_complex=True, center=True, pad_mode="constant").abs()
            est_mag = est_mag.reshape(batch, sources, channels, *est_mag.shape[-2:])
            tgt_mag = tgt_mag.reshape(batch, sources, channels, *tgt_mag.shape[-2:])
            diff_norm = torch.linalg.vector_norm((est_mag - tgt_mag).flatten(start_dim=2), dim=-1)
            target_norm = torch.linalg.vector_norm(tgt_mag.flatten(start_dim=2), dim=-1).clamp_min(1e-4)
            convergence = torch.where(active, diff_norm / target_norm, torch.zeros_like(diff_norm))
            log_magnitude = (torch.log1p(est_mag) - torch.log1p(tgt_mag)).abs().mean(dim=(-3, -2, -1))
            total = total + (convergence + log_magnitude).mean()
        return total / len(fft_sizes)


def si_sdr_loss(estimate: Tensor, target: Tensor, silence_threshold_db: float = -60.0) -> Tensor:
    """Negative active-stem SI-SDR, clipped to [-30,30] dB for stability."""
    _check_pair(estimate, target)
    with torch.autocast(device_type=estimate.device.type, enabled=False):
        active = target.float().square().mean(dim=(-2, -1)) > 10.0 ** (silence_threshold_db / 10.0)
        values = si_sdr_values(estimate, target).clamp(-30.0, 30.0)
        # Tensor-only aggregation keeps the all-silent result attached to autograd.
        return -(values * active).sum() / active.sum().clamp_min(1)


def separation_loss(estimate: Tensor, target: Tensor, mixture: Tensor,
                    spectral_weight: float = 0.05, si_sdr_weight: float = 0.01,
                    consistency_weight: float = 0.1,
                    fft_sizes: Sequence[int] = (512, 1024, 2048, 4096)) -> tuple[Tensor, dict[str, Tensor]]:
    """Return the scalar objective and detached component values for logging.

    All waveforms are unnormalized floating-point amplitudes. Supply the actual
    augmented mixture corresponding to ``target``. Weights are starting points,
    to be selected on a fixed validation set rather than called universally best.
    """
    _check_pair(estimate, target)
    if mixture.shape != (target.shape[0], target.shape[2], target.shape[3]):
        raise ValueError("mixture must have [batch, channel, samples] matching the targets")
    if min(spectral_weight, si_sdr_weight, consistency_weight) < 0:
        raise ValueError("loss weights must be nonnegative")
    with torch.autocast(device_type=estimate.device.type, enabled=False):
        estimate, target, mixture = estimate.float(), target.float(), mixture.float()
        scales = _source_scales(target, mixture)
        source_l1 = ((estimate - target).abs().mean(dim=(-2, -1)) / scales).mean()
        mixture_scale = mixture.square().mean(dim=(-2, -1)).clamp_min(1e-8).sqrt()
        mixture_l1 = ((estimate.sum(dim=1) - mixture).abs().mean(dim=(-2, -1)) / mixture_scale).mean()
        inactive = target.square().mean(dim=(-2, -1)) <= 1e-6
        silence_per_source = estimate.abs().mean(dim=(-2, -1)) / scales
        silence = (silence_per_source * inactive).sum() / inactive.sum().clamp_min(1)
        zero = source_l1 * 0
        spectral = multi_resolution_stft_loss(estimate, target, mixture, fft_sizes) if spectral_weight else zero
        si_sdr = si_sdr_loss(estimate, target) if si_sdr_weight else zero
        total = (source_l1 + consistency_weight * mixture_l1 + spectral_weight * spectral
                 + si_sdr_weight * (si_sdr + silence))
        parts = {"source_l1": source_l1, "mixture_l1": mixture_l1,
                 "spectral": spectral, "si_sdr": si_sdr, "silence": silence}
        return total, {name: value.detach() for name, value in parts.items()}


In [ ]:
%%writefile audio_sep/metrics.py
"""Small, transparent metrics; these are not the official MUSDB BSS Eval SDR.

SNR here is 10*log10(target energy / reconstruction error energy). SI-SDR uses
one common scale for the stereo pair after removing each channel's temporal mean.
All results have shape [batch, source]. Exclude entries with ``active=False``
from reported means; their dB values are zero placeholders, not measurements.
"""

from __future__ import annotations

import torch
from torch import Tensor


def _check_pair(estimate: Tensor, target: Tensor) -> None:
    if estimate.ndim != 4 or estimate.shape != target.shape or estimate.shape[-1] < 1:
        raise ValueError("estimate and target must have equal [batch, source, channel, samples] shapes")


def si_sdr_values(estimate: Tensor, target: Tensor, eps: float = 1e-8) -> Tensor:
    """Stereo SI-SDR [B,S], with float32 arithmetic and bounded numerical floors."""
    _check_pair(estimate, target)
    estimate = estimate.float()
    target = target.float()
    estimate = (estimate - estimate.mean(dim=-1, keepdim=True)).flatten(start_dim=2)
    target = (target - target.mean(dim=-1, keepdim=True)).flatten(start_dim=2)
    target_power = target.square().mean(dim=-1, keepdim=True)
    gain = (estimate * target).mean(dim=-1, keepdim=True) / target_power.clamp_min(eps)
    projected = gain * target
    residual = estimate - projected
    projection_power = projected.square().mean(dim=-1)
    residual_power = residual.square().mean(dim=-1)
    return 10.0 * torch.log10((projection_power + eps) / (residual_power + eps))


def metrics_per_source(estimate: Tensor, target: Tensor,
                       silence_threshold_db: float = -60.0) -> dict[str, Tensor]:
    """Return metrics in the tensor/model source order, including silence leakage.

    ``silent_estimate_energy`` is linear mean-square amplitude on inactive stems
    and zero on active ones. Report its mean only over inactive entries.
    Silence threshold is power in dBFS; -60 corresponds to an RMS of 0.001.
    """
    _check_pair(estimate, target)
    with torch.autocast(device_type=estimate.device.type, enabled=False):
        estimate, target = estimate.float(), target.float()
        power = target.square().mean(dim=(-2, -1))
        active = power > 10.0 ** (silence_threshold_db / 10.0)
        error_power = (estimate - target).square().mean(dim=(-2, -1))
        snr = 10.0 * torch.log10((power + 1e-8) / (error_power + 1e-8))
        si_sdr = si_sdr_values(estimate, target)
        estimate_power = estimate.square().mean(dim=(-2, -1))
        return {
            "snr_db": torch.where(active, snr, torch.zeros_like(snr)),
            "si_sdr_db": torch.where(active, si_sdr, torch.zeros_like(si_sdr)),
            "active": active,
            "silent_estimate_energy": torch.where(active, torch.zeros_like(estimate_power), estimate_power),
        }


In [ ]:
%%writefile audio_sep/model.py
"""A stereo waveform U-Net intended for training a compact separator from scratch.

This is a new architecture: original ``DemucsLite`` checkpoints are not compatible.
It does not contain pretrained weights and does not promise state-of-the-art quality.
"""

from __future__ import annotations

import math
from typing import Sequence

import torch
from torch import Tensor, nn
from torch.nn import functional as F


DEFAULT_SOURCES = ("vocals", "drums", "bass", "other")


class ResidualDConv(nn.Module):
    """Compressed, dilated residual convolutions without batch statistics."""

    def __init__(self, channels: int, depth: int = 2, compression: int = 4):
        super().__init__()
        hidden = max(8, channels // compression)
        self.blocks = nn.ModuleList()
        self.scales = nn.ParameterList()
        for index in range(depth):
            dilation = 2**index
            self.blocks.append(
                nn.Sequential(
                    nn.Conv1d(channels, hidden, kernel_size=1),
                    # One group also works for batch=1 and a single bottleneck frame.
                    nn.GroupNorm(1, hidden),
                    nn.GELU(),
                    nn.Conv1d(hidden, hidden, 3, padding=dilation, dilation=dilation),
                    nn.GroupNorm(1, hidden),
                    nn.GELU(),
                    nn.Conv1d(hidden, 2 * channels, kernel_size=1),
                    nn.GLU(dim=1),
                )
            )
            self.scales.append(nn.Parameter(torch.full((1, channels, 1), 1e-3)))

    def forward(self, x: Tensor) -> Tensor:
        for block, scale in zip(self.blocks, self.scales):
            x = x + scale * block(x)
        return x


class EncoderBlock(nn.Module):
    def __init__(self, input_channels: int, output_channels: int,
                 kernel_size: int, stride: int, dconv_depth: int):
        super().__init__()
        self.conv = nn.Conv1d(input_channels, output_channels, kernel_size, stride)
        self.context = ResidualDConv(output_channels, dconv_depth)
        self.gate = nn.Conv1d(output_channels, 2 * output_channels, 1)

    def forward(self, x: Tensor) -> Tensor:
        x = F.gelu(self.conv(x))
        return F.glu(self.gate(self.context(x)), dim=1)


class DecoderBlock(nn.Module):
    def __init__(self, input_channels: int, output_channels: int,
                 kernel_size: int, stride: int, dconv_depth: int, last: bool):
        super().__init__()
        self.context = ResidualDConv(input_channels, dconv_depth)
        self.gate = nn.Conv1d(input_channels, 2 * input_channels, 3, padding=1)
        self.upsample = nn.ConvTranspose1d(input_channels, output_channels, kernel_size, stride)
        self.last = last

    def forward(self, x: Tensor, skip: Tensor) -> Tensor:
        x = (x + skip) * (2.0**-0.5)
        x = F.glu(self.gate(self.context(x)), dim=1)
        x = self.upsample(x)
        return x if self.last else F.gelu(x)


class DemucsLiteV2(nn.Module):
    """Predict named stereo stems with the same number of samples as the input.

    Input: ``[batch, audio_channels, samples]``. Output:
    ``[batch, len(sources), audio_channels, samples]`` in ``sources`` order.
    Normalization uses one shared mean/scale per mixture, preserving stereo gains.
    Optional mixture consistency distributes the remaining residual equally. Leave
    it disabled when measuring whether the model learned to reconstruct mixtures.
    ``lstm_layers=0`` disables the recurrent bottleneck for a cheaper deployment.
    """

    def __init__(
        self,
        sources: Sequence[str] = DEFAULT_SOURCES,
        audio_channels: int = 2,
        sample_rate: int = 44100,
        channels: int = 32,
        depth: int = 5,
        growth: float = 2.0,
        max_channels: int = 512,
        kernel_size: int = 8,
        stride: int = 4,
        lstm_layers: int = 2,
        dconv_depth: int = 2,
        normalize: bool = True,
        mixture_consistency: bool = False,
    ):
        super().__init__()
        if isinstance(sources, (str, int)) or not sources:
            raise ValueError("sources must be a nonempty sequence of unique stem names")
        self.sources = tuple(sources)
        if len(set(self.sources)) != len(self.sources) or not all(isinstance(s, str) for s in self.sources):
            raise ValueError("sources must contain unique strings")
        if min(audio_channels, sample_rate, channels, depth, max_channels, kernel_size, stride) < 1:
            raise ValueError("channel counts, depth, sample rate, kernel and stride must be positive")
        if kernel_size < stride or growth < 1 or lstm_layers < 0 or dconv_depth < 0:
            raise ValueError("require kernel_size >= stride, growth >= 1 and nonnegative block depths")
        self.audio_channels = audio_channels
        self.sample_rate = sample_rate
        self.channels = channels
        self.depth = depth
        self.growth = growth
        self.max_channels = max_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.lstm_layers = lstm_layers
        self.dconv_depth = dconv_depth
        self.normalize = normalize
        self.mixture_consistency = mixture_consistency

        channel_sizes = [min(max_channels, int(channels * growth**index)) for index in range(depth)]
        self.encoders = nn.ModuleList()
        input_channels = audio_channels
        for index, output_channels in enumerate(channel_sizes):
            # Avoid costly context convolutions on the longest, shallow features.
            block_depth = dconv_depth if index >= 2 else 0
            self.encoders.append(EncoderBlock(input_channels, output_channels, kernel_size, stride, block_depth))
            input_channels = output_channels

        bottleneck_channels = channel_sizes[-1]
        if lstm_layers:
            hidden = max(1, bottleneck_channels // 2)
            self.lstm = nn.LSTM(bottleneck_channels, hidden, num_layers=lstm_layers,
                                bidirectional=True, batch_first=True)
            self.lstm_projection = nn.Linear(2 * hidden, bottleneck_channels)
        else:
            self.lstm = None
            self.lstm_projection = None

        self.decoders = nn.ModuleList()
        for index in reversed(range(depth)):
            output_channels = channel_sizes[index - 1] if index else len(self.sources) * audio_channels
            self.decoders.append(DecoderBlock(channel_sizes[index], output_channels, kernel_size,
                                              stride, dconv_depth if index >= 2 else 0, index == 0))

    def export_config(self) -> dict:
        """JSON-compatible configuration, including the exact stem order."""
        return {
            "sources": list(self.sources), "audio_channels": self.audio_channels,
            "sample_rate": self.sample_rate, "channels": self.channels, "depth": self.depth,
            "growth": self.growth, "max_channels": self.max_channels,
            "kernel_size": self.kernel_size, "stride": self.stride,
            "lstm_layers": self.lstm_layers, "dconv_depth": self.dconv_depth,
            "normalize": self.normalize, "mixture_consistency": self.mixture_consistency,
        }

    @property
    def samplerate(self) -> int:
        """Match the metadata name used by official Demucs inference utilities."""
        return self.sample_rate

    def valid_length(self, length: int) -> int:
        """Next input length that all valid strided convolutions reconstruct."""
        if length < 1:
            raise ValueError("audio must contain at least one sample")
        for _ in range(self.depth):
            length = max(1, math.ceil((length - self.kernel_size) / self.stride) + 1)
        for _ in range(self.depth):
            length = (length - 1) * self.stride + self.kernel_size
        return int(length)

    def forward(self, mixture: Tensor) -> Tensor:
        if mixture.ndim != 3 or mixture.shape[1] != self.audio_channels:
            raise ValueError(f"expected [batch, {self.audio_channels}, samples], got {tuple(mixture.shape)}")
        original_length = mixture.shape[-1]
        padded_length = self.valid_length(original_length)
        if self.normalize:
            mean = mixture.mean(dim=(1, 2), keepdim=True)
            centered = mixture - mean
            scale = centered.square().mean(dim=(1, 2), keepdim=True).clamp_min(1e-6).sqrt()
            x = centered / scale
        else:
            mean = mixture.new_zeros((mixture.shape[0], 1, 1))
            scale = mixture.new_ones((mixture.shape[0], 1, 1))
            x = mixture
        left_padding = (padded_length - original_length) // 2
        right_padding = padded_length - original_length - left_padding
        x = F.pad(x, (left_padding, right_padding))
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
        if self.lstm is not None:
            x, _ = self.lstm(x.transpose(1, 2))
            x = self.lstm_projection(x).transpose(1, 2)
        for decoder, skip in zip(self.decoders, reversed(skips)):
            x = decoder(x, skip)
        x = x[..., left_padding:left_padding + original_length]
        x = x.reshape(mixture.shape[0], len(self.sources), self.audio_channels, original_length)
        # Adding the mixture mean to every stem would multiply its DC by S.
        x = x * scale[:, None] + mean[:, None] / len(self.sources)
        if self.mixture_consistency:
            x = x + (mixture - x.sum(dim=1))[:, None] / len(self.sources)
        return x


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


In [ ]:
%%writefile audio_sep/pretrained.py
"""Train one HTDemucs backbone, preserving its source order and context length."""
import torch
from torch import nn
from torch.nn import functional as F

from .data import DEFAULT_SOURCES


class PretrainedSeparator(nn.Module):
    def __init__(self, name="htdemucs", sources=DEFAULT_SOURCES, segment=None,
                 init_args=None, init_kwargs=None):
        super().__init__()
        if init_args is not None and init_kwargs is not None:
            from demucs.htdemucs import HTDemucs
            backbone = HTDemucs(*init_args, **init_kwargs)
        else:
            from .compat import official_model
            backbone = official_model(name)
        if hasattr(backbone, "models"):
            if len(backbone.models) != 1:
                raise ValueError("Fine-tune one htdemucs backbone; use htdemucs_ft for inference.")
            backbone = backbone.models[0]
        self.backbone = backbone
        self.name = name
        self.sources = tuple(sources)
        if set(self.sources) != set(backbone.sources):
            raise ValueError("Source names do not match pretrained backbone.")
        self.audio_channels = backbone.audio_channels
        self.samplerate = backbone.samplerate
        self.segment = float(backbone.segment if segment is None else segment)
        self.backbone.segment = self.segment
        self.order = [backbone.sources.index(s) for s in self.sources]

    def forward(self, mixture):
        length = mixture.shape[-1]
        context = round(self.segment * self.samplerate)
        if length > context:
            raise ValueError("Input exceeds HTDemucs training context; use split inference.")
        # Official Demucs input conditioning; one shared transform for every stem.
        reference = mixture.mean(dim=1, keepdim=True)
        mean = reference.mean(dim=-1, keepdim=True)
        std = reference.std(dim=-1, keepdim=True, unbiased=False).clamp_min(1e-5)
        x = F.pad((mixture - mean) / std, (0, context - length))
        # HTDemucs.forward mutates segment in train mode. Keep the original context.
        y = self.backbone(x)[:, self.order, ..., :length]
        self.backbone.segment = self.segment
        return y * std[:, None] + mean[:, None] / len(self.sources)

    def export_config(self):
        args, kwargs = self.backbone._init_args_kwargs
        return {"name": self.name, "sources": list(self.sources), "segment": self.segment,
                "init_args": args, "init_kwargs": kwargs}


In [ ]:
%%writefile audio_sep/train.py
"""Training entry point with resumable state and selection on validation SNR."""
import argparse
import copy
import csv
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import DEFAULT_SOURCES, MUSDBDataset, load_manifest
from .losses import separation_loss
from .metrics import metrics_per_source


# A bounded recovery guard, not a claim that this is a universally optimal limit.
MAX_CONSECUTIVE_AMP_OVERFLOWS = 8


def restore_scaler_state(scaler, saved_state):
    """Restore AMP state only when the current run uses an enabled scaler.

    AMP -> FP32 deliberately ignores the saved scaler. FP32 -> AMP starts with
    a fresh scale because disabled scalers save an empty state dictionary.
    """
    if not scaler.is_enabled():
        return "disabled"
    if saved_state:
        scaler.load_state_dict(saved_state)
        return "restored"
    return "fresh"


def checked_optimizer_step(model, optimizer, scaler, *, step=0, ema=None,
                           ema_decay=0.999, max_norm=5.0):
    """Unscale once, recover AMP overflow, and update EMA after real steps only.

    The norm is accumulated in float64: a float32 sum of squares can overflow
    even when every gradient entry is finite. This distinguishes that case from
    the nonfinite entries GradScaler detects and skips. Returned diagnostics do
    not infer which operation originally produced a nonfinite gradient.
    """
    if max_norm <= 0:
        raise ValueError("max_norm must be positive")
    scaler.unscale_(optimizer)
    parameters = [parameter for parameter in model.parameters() if parameter.grad is not None]
    if not parameters:
        raise ValueError("No gradients to update")
    finite_entries = bool(torch.stack([torch.isfinite(parameter.grad).all()
                                      for parameter in parameters]).all())
    scale_before = float(scaler.get_scale())
    if not finite_entries:
        if not scaler.is_enabled():
            optimizer.zero_grad(set_to_none=True)
            raise FloatingPointError("Nonfinite FP32 gradient entries; inspect loss/data/model.")
        # unscale_ recorded the inf/NaN entries. GradScaler skips optimizer.step
        # and update backs off its scale; do not clip or update EMA in this case.
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        return {"stepped": False, "step": step, "gradient_entries_finite": False,
                "grad_norm": None, "scale_before": scale_before,
                "scale_after": float(scaler.get_scale())}
    with torch.no_grad():
        norms = torch.stack([torch.linalg.vector_norm(parameter.grad, dtype=torch.float64)
                             for parameter in parameters])
        norm = torch.linalg.vector_norm(norms)
        if not bool(torch.isfinite(norm)):
            optimizer.zero_grad(set_to_none=True)
            raise FloatingPointError("Nonfinite float64 gradient norm despite finite entries; no optimizer step.")
        coefficient = (max_norm / (norm + 1e-6)).clamp(max=1.0)
        for parameter in parameters:
            parameter.grad.mul_(coefficient)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    step += 1
    if ema is not None:
        decay = min(ema_decay, (1 + step) / (10 + step))
        with torch.no_grad():
            for average, current in zip(ema.parameters(), model.parameters()):
                average.lerp_(current.detach(), 1 - decay)
            for average, current in zip(ema.buffers(), model.buffers()):
                average.copy_(current)
    return {"stepped": True, "step": step, "gradient_entries_finite": True,
            "grad_norm": float(norm), "scale_before": scale_before,
            "scale_after": float(scaler.get_scale())}


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def atomic_save(obj, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, temporary)
    os.replace(temporary, path)


def build_model(kind, config):
    if kind == "lite":
        from .model import DemucsLiteV2
        return DemucsLiteV2(**config)
    from .pretrained import PretrainedSeparator
    return PretrainedSeparator(**config)


@torch.inference_mode()
def validate(model, loader, device, amp, loss_config):
    model.eval()
    values, masks, si_values = [], [], []
    total, count = 0.0, 0
    for mix, target in tqdm(loader, desc="Validation", leave=False):
        mix, target = mix.to(device), target.to(device)
        with torch.autocast(device.type, enabled=amp):
            estimate = model(mix)
        loss, _ = separation_loss(estimate, target, mix, **loss_config)
        metrics = metrics_per_source(estimate, target)
        values.append(metrics["snr_db"].cpu())
        si_values.append(metrics["si_sdr_db"].cpu())
        masks.append(metrics["active"].cpu())
        total += loss.item() * len(mix)
        count += len(mix)
    values, si_values, masks = torch.cat(values), torch.cat(si_values), torch.cat(masks)
    # Equal track weight, then equal source weight; silent crops are excluded.
    n_tracks = len(loader.dataset.records)
    n_crops = count // n_tracks
    mask = masks.reshape(n_tracks, n_crops, -1)
    per_track = []
    for tensor in (values, si_values):
        x = tensor.reshape(n_tracks, n_crops, -1)
        active_count = mask.sum(1)
        track_score = (x * mask).sum(1) / active_count.clamp_min(1)
        track_score[active_count == 0] = float("nan")
        per_track.append(torch.nanmean(track_score, dim=0))
    snr, si = per_track
    if not torch.isfinite(snr).any():
        raise ValueError("No active validation targets.")
    result = {"loss": total / count, "snr_db": float(torch.nanmean(snr)),
              "si_sdr_db": float(torch.nanmean(si))}
    for i, source in enumerate(model.sources):
        result[f"snr_{source}_db"] = float(snr[i])
        result[f"si_sdr_{source}_db"] = float(si[i])
        result[f"active_{source}"] = float(masks[:, i].float().mean())
    return result


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--manifest", required=True)
    p.add_argument("--run-dir", required=True)
    p.add_argument("--kind", choices=["lite", "pretrained"], default="lite")
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--batch-size", type=int, default=1)
    p.add_argument("--accumulate", type=int, default=4)
    p.add_argument("--segment", type=float, default=6)
    p.add_argument("--train-crops", type=int, default=32)
    p.add_argument("--val-crops", type=int, default=8)
    p.add_argument("--channels", type=int, default=32)
    p.add_argument("--depth", type=int, default=5)
    p.add_argument("--lr", type=float)
    p.add_argument("--weight-decay", type=float, default=1e-4)
    p.add_argument("--spectral-weight", type=float, default=0.05)
    p.add_argument("--si-sdr-weight", type=float, default=0.01)
    p.add_argument("--ema-decay", type=float, default=0.999)
    p.add_argument("--patience", type=int, default=15)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume", action="store_true")
    p.add_argument("--no-amp", action="store_true")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = p.parse_args()
    if min(args.epochs, args.batch_size, args.accumulate, args.train_crops, args.val_crops) <= 0:
        p.error("Epochs, batch size, accumulation and crops must be positive.")
    if not 0 <= args.ema_decay < 1:
        p.error("EMA decay must be in [0,1).")
    seed_all(args.seed)
    device = torch.device(args.device)
    amp = device.type == "cuda" and not args.no_amp
    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    config = ({"sources": DEFAULT_SOURCES, "channels": args.channels, "depth": args.depth}
              if args.kind == "lite" else {"name": "htdemucs", "sources": DEFAULT_SOURCES})
    saved = None
    if args.resume:
        saved = torch.load(run_dir / "last.pth", map_location="cpu", weights_only=False)
        if saved["model_kind"] != args.kind:
            raise ValueError("Cannot resume a different architecture.")
        config = saved["model_config"]
    elif (run_dir / "last.pth").exists():
        raise FileExistsError("Run already exists. Use --resume or a new run-dir.")
    model = build_model(args.kind, config).to(device)
    if args.kind == "pretrained":
        args.segment = min(args.segment, model.segment)
    learning_rate = args.lr if args.lr is not None else (3e-4 if args.kind == "lite" else 1e-5)
    loss_config = {"spectral_weight": args.spectral_weight, "si_sdr_weight": args.si_sdr_weight,
                   "consistency_weight": 0.1}
    records = load_manifest(args.manifest)
    train_records = [r for r in records if r["subset"] == "train"]
    val_records = [r for r in records if r["subset"] == "val"]
    if not train_records or not val_records:
        raise ValueError("Manifest must contain train and val tracks.")
    train_data = MUSDBDataset(train_records, sources=model.sources, segment_seconds=args.segment,
                             crops_per_track=args.train_crops, training=True, seed=args.seed)
    val_data = MUSDBDataset(val_records, sources=model.sources, segment_seconds=args.segment,
                           training=False, validation_crops=args.val_crops, seed=args.seed)
    generator = torch.Generator().manual_seed(args.seed)
    loader_options = {"num_workers": args.workers, "pin_memory": device.type == "cuda",
                      "persistent_workers": False}
    train_loader = DataLoader(train_data, args.batch_size, shuffle=True, generator=generator, **loader_options)
    val_loader = DataLoader(val_data, args.batch_size, shuffle=False, **loader_options)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5,
                                                         patience=5, min_lr=learning_rate / 30)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    ema = copy.deepcopy(model).eval() if args.ema_decay else None
    if ema is not None:
        ema.requires_grad_(False)
    start, best, stale, step = 1, -math.inf, 0, 0
    history = []
    amp_overflows_total, consecutive_overflows = 0, 0
    # Resume only identical data/loss/training settings; number of epochs may grow.
    settings = vars(args).copy()
    settings["lr"] = learning_rate
    settings["amp_enabled"] = amp
    settings["precision_mode"] = "cuda_amp" if amp else "fp32"
    if saved:
        for key in ("segment", "train_crops", "val_crops", "batch_size", "accumulate", "seed",
                    "ema_decay", "weight_decay", "spectral_weight", "si_sdr_weight", "lr"):
            if saved["training_config"][key] != settings[key]:
                raise ValueError(f"Resume config changed: {key}. Start a new run.")
        if saved["manifest"] != records or saved["loss_config"] != loss_config:
            raise ValueError("Resume requires identical manifest and loss.")
        model.load_state_dict(saved["model_state_dict"])
        if ema is not None:
            ema.load_state_dict(saved["ema_state_dict"])
        optimizer.load_state_dict(saved["optimizer_state_dict"])
        scheduler.load_state_dict(saved["scheduler_state_dict"])
        scaler_restoration = restore_scaler_state(scaler, saved.get("scaler_state_dict", {}))
        previous_amp = saved["training_config"].get("amp_enabled", bool(saved.get("scaler_state_dict")))
        settings["resume_precision_from"] = "cuda_amp" if previous_amp else "fp32"
        print(f"Resume precision: {settings['resume_precision_from']} -> {settings['precision_mode']}; "
              f"scaler {scaler_restoration}", flush=True)
        start, best, stale, step = saved["epoch"] + 1, saved["best_snr_db"], saved["stale"], saved["step"]
        amp_overflows_total = saved.get("amp_overflows_total", 0)
        consecutive_overflows = saved.get("consecutive_amp_overflows", 0)
        history = saved["history"]
        torch.set_rng_state(saved["rng_torch"])
        random.setstate(saved["rng_python"])
        np.random.set_state(saved["rng_numpy"])
        generator.set_state(saved["rng_loader"])
        if device.type == "cuda" and saved.get("rng_cuda") is not None:
            torch.cuda.set_rng_state_all(saved["rng_cuda"])
    (run_dir / "config.json").write_text(json.dumps(settings, indent=2))

    def checkpoint(epoch, selected="raw"):
        return {"format_version": 2, "model_kind": args.kind, "model_config": model.export_config(),
                "sources": list(model.sources), "sample_rate": 44100,
                "model_state_dict": model.state_dict(),
                "ema_state_dict": ema.state_dict() if ema is not None else None,
                "selected_weights": selected, "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(),
                "epoch": epoch, "step": step, "best_snr_db": best, "stale": stale,
                "amp_overflows_total": amp_overflows_total,
                "consecutive_amp_overflows": consecutive_overflows,
                "training_config": settings, "loss_config": loss_config, "history": history,
                "manifest": records, "rng_torch": torch.get_rng_state(),
                "rng_python": random.getstate(), "rng_numpy": np.random.get_state(),
                "rng_loader": generator.get_state(),
                "rng_cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None}

    # Retain pretrained baseline if fine-tuning makes validation worse.
    if not saved and args.kind == "pretrained":
        baseline = validate(model, val_loader, device, amp, loss_config)
        best = baseline["snr_db"]
        history.append({"epoch": 0, "train_loss": None, "lr": learning_rate, "selected": "raw",
                        "precision_mode": settings["precision_mode"], "amp_skipped_groups": 0,
                        "amp_overflows_total": 0, "amp_scale": float(scaler.get_scale()), **baseline})
        atomic_save(checkpoint(0), run_dir / "best.pth")
        export_best(run_dir)
    for epoch in range(start, args.epochs + 1):
        train_data.set_epoch(epoch)
        model.train()
        total, count = 0.0, 0
        skipped_groups = 0
        optimizer.zero_grad(set_to_none=True)
        for batch_idx, (mix, target) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{args.epochs}")):
            mix, target = mix.to(device, non_blocking=True), target.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=amp):
                estimate = model(mix)
            loss, _ = separation_loss(estimate, target, mix, **loss_config)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite loss. Retry with --no-amp and inspect dataset.")
            group_start = (batch_idx // args.accumulate) * args.accumulate
            divisor = min(args.accumulate, len(train_loader) - group_start)
            scaler.scale(loss / divisor).backward()
            if (batch_idx + 1) % args.accumulate == 0 or batch_idx + 1 == len(train_loader):
                update = checked_optimizer_step(model, optimizer, scaler, step=step,
                                                ema=ema, ema_decay=args.ema_decay)
                step = update["step"]
                if update["stepped"]:
                    consecutive_overflows = 0
                else:
                    skipped_groups += 1
                    amp_overflows_total += 1
                    consecutive_overflows += 1
                    print(f"AMP nonfinite gradient entries: skipped group at epoch {epoch}, "
                          f"batch {batch_idx + 1}; scale {update['scale_before']:g} -> "
                          f"{update['scale_after']:g}; consecutive {consecutive_overflows}/"
                          f"{MAX_CONSECUTIVE_AMP_OVERFLOWS}; optimizer/EMA unchanged", flush=True)
                    if consecutive_overflows >= MAX_CONSECUTIVE_AMP_OVERFLOWS:
                        raise FloatingPointError(
                            f"AMP has {consecutive_overflows} consecutive nonfinite-gradient groups; "
                            "stopped without changing the best checkpoint. Resume --no-amp "
                            "or inspect loss/data/model.")
            total += loss.item() * len(mix)
            count += len(mix)
        raw_metrics = validate(model, val_loader, device, amp, loss_config)
        selected, val = "raw", raw_metrics
        if ema is not None:
            ema_metrics = validate(ema, val_loader, device, amp, loss_config)
            if ema_metrics["snr_db"] > val["snr_db"]:
                selected, val = "ema", ema_metrics
        learning_rate_used = optimizer.param_groups[0]["lr"]
        scheduler.step(val["snr_db"])
        improved = val["snr_db"] > best + 1e-4
        best, stale = (val["snr_db"], 0) if improved else (best, stale + 1)
        history.append({"epoch": epoch, "train_loss": total / count, "lr": learning_rate_used,
                        "selected": selected, "precision_mode": settings["precision_mode"],
                        "amp_skipped_groups": skipped_groups,
                        "amp_overflows_total": amp_overflows_total,
                        "amp_scale": float(scaler.get_scale()), **val})
        state = checkpoint(epoch, selected)
        atomic_save(state, run_dir / "last.pth")
        if improved:
            atomic_save(state, run_dir / "best.pth")
            # Keep a deployable export even if a later epoch/runtime is interrupted.
            export_best(run_dir)
        with (run_dir / "history.csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(history[-1]))
            writer.writeheader()
            writer.writerows(history)
        print(json.dumps(history[-1]), flush=True)
        if args.patience and stale >= args.patience:
            print("Early stopping: validation SNR has stopped improving.")
            break
    export_best(run_dir)


def export_best(run_dir):
    run_dir = Path(run_dir)
    best = torch.load(run_dir / "best.pth", map_location="cpu", weights_only=False)
    state = best["ema_state_dict"] if best["selected_weights"] == "ema" else best["model_state_dict"]
    export = {k: best[k] for k in ("format_version", "model_kind", "model_config", "sources", "sample_rate", "epoch", "best_snr_db")}
    export["model_state_dict"] = state
    export["ema_state_dict"] = None
    export["selected_weights"] = best["selected_weights"]
    atomic_save(export, run_dir / "best_model.pth")
    print(f"Exported {run_dir / 'best_model.pth'} (epoch {best['epoch']}, val SNR {best['best_snr_db']:.3f} dB)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_data.py
"""Synthetic tests for alignment, held-out songs, and augmentation physics."""

import json
from pathlib import Path
import tempfile
import unittest
import warnings

import numpy as np
import soundfile as sf
import torch

from audio_sep.data import DEFAULT_SOURCES, KNOWN_LEFT_MIX_ERRATUM, MUSDBDataset, load_manifest, scan_dataset


class DataTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.root = Path(self.temp.name)
        self.addCleanup(self.temp.cleanup)

    def track(self, name="Synthetic Song", split="train", seed=7, subtype="FLOAT", frames=3000):
        path = self.root / split / name
        path.mkdir(parents=True)
        rng = np.random.default_rng(seed)
        targets = []
        for index, source in enumerate(DEFAULT_SOURCES):
            mono = rng.normal(0, 0.025 * (index + 1), frames).astype(np.float32)
            # Unequal channels detect accidental mono conversion.
            stem = np.stack([mono, mono * 0.37], axis=-1)
            sf.write(path / f"{source}.wav", stem, 44100, subtype=subtype)
            targets.append(stem)
        sf.write(path / "mixture.wav", np.stack(targets).sum(axis=0), 44100, subtype=subtype)
        return path

    def scan(self, **kwargs):
        return scan_dataset(self.root, strict=False, **kwargs)

    def test_official_song_split_has_no_crop_leakage_and_manifest_relocates(self):
        self.track()
        self.track(name="Actions - One Minute Smile", seed=8)
        self.track(name="Held-out Test Song", split="test", seed=9)
        output = self.root / "manifest.json"
        records = self.scan(output=output)
        splits = {record["track_name"]: record["subset"] for record in records}
        self.assertEqual(splits, {"Synthetic Song": "train", "Actions - One Minute Smile": "val", "Held-out Test Song": "test"})
        self.assertEqual(records, load_manifest(output))
        relocated_root = (self.root / "relocated").resolve()
        relocated = load_manifest(output, root=relocated_root)
        for record in relocated:
            self.assertEqual(Path(record["path"]).relative_to(relocated_root), Path(record["relative_path"]))
        audit = json.loads((self.root / "manifest.audit.json").read_text())
        self.assertEqual(audit["counts"], {"train": 1, "val": 1, "test": 1})
        self.assertFalse(audit["errors"])
        self.assertIn("active_fraction", audit["tracks"][0]["probes"][0])
        with self.assertRaisesRegex(ValueError, "Expected 100"):
            scan_dataset(self.root)

    def test_alignment_sample_rate_and_stereo_are_checked(self):
        path = self.track()
        sf.write(path / "bass.wav", np.zeros((2999, 2)), 44100, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "bass.wav: 2999 frames"):
            self.scan()
        sf.write(path / "bass.wav", np.zeros((3000, 1)), 22050, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "expected 44100 Hz stereo"):
            self.scan()

    def test_quantization_rounding_is_tolerated_but_bad_mixture_fails(self):
        path = self.track(subtype="PCM_16")
        with warnings.catch_warnings(record=True) as caught:
            self.scan()
        self.assertEqual(len(caught), 0)
        sf.write(path / "mixture.wav", np.full((3000, 2), 0.25), 44100, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "mixture/stem residual"):
            self.scan()

    def test_nonfinite_probe_and_train_test_duplicates_fail(self):
        path = self.track()
        bad = np.zeros((3000, 2), np.float32)
        bad[0, 0] = np.nan
        sf.write(path / "vocals.wav", bad, 44100, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "non-finite"):
            self.scan()
        sf.write(path / "vocals.wav", np.zeros((3000, 2)), 44100, subtype="FLOAT")
        self.track(split="test")
        with self.assertRaisesRegex(ValueError, "Training/test track-name overlap"):
            self.scan()

    def test_known_test_left_erratum_is_audited_and_original_audio_is_preserved(self):
        path = self.track(name="PR - Oh No", split="test")
        mixture, sr = sf.read(path / "mixture.wav", dtype="float32", always_2d=True)
        mixture[:, 0] += np.float32(0.25)
        sf.write(path / "mixture.wav", mixture, sr, subtype="FLOAT")
        originals = {name: (path / f"{name}.wav").read_bytes() for name in ("mixture", *DEFAULT_SOURCES)}
        output = self.root / "manifest.json"
        with warnings.catch_warnings(record=True) as caught:
            records = self.scan(output=output)
        self.assertTrue(any("known MUSDB erratum on left channel" in str(item.message) for item in caught))
        self.assertEqual(len(records), 1)
        self.assertEqual(records[0]["subset"], "test")
        self.assertEqual(records[0]["known_issue"], KNOWN_LEFT_MIX_ERRATUM)
        self.assertEqual(records[0]["residual_policy"], "warn_known_artistic_mix")
        self.assertGreater(records[0]["max_probe_residual_rms_by_channel"]["left"], 0.2)
        self.assertLess(records[0]["max_probe_residual_rms_by_channel"]["right"], 1e-6)
        self.assertEqual(records, load_manifest(output))
        audit = json.loads((self.root / "manifest.audit.json").read_text())
        self.assertFalse(audit["errors"])
        self.assertEqual(audit["tracks"][0]["known_issue"], KNOWN_LEFT_MIX_ERRATUM)
        self.assertTrue(audit["known_errata_policy"]["original_mixture_stems_preserved"])
        channels = audit["tracks"][0]["probes"][0]["residual_by_channel"]
        self.assertGreater(channels["left"]["residual_rms"], channels["left"]["failure_floor"])
        self.assertLess(channels["right"]["residual_rms"], channels["right"]["failure_floor"])
        dataset = MUSDBDataset(records, training=False, segment_seconds=0.02, validation_crops=1)
        actual_mix, actual_stems = dataset[0]
        offset = (3000 - 882) // 2
        torch.testing.assert_close(actual_mix, torch.from_numpy(mixture[offset:offset + 882].T.copy()), rtol=0, atol=0)
        expected_stems = torch.stack([
            torch.from_numpy(sf.read(path / f"{name}.wav", start=offset, frames=882,
                                     dtype="float32", always_2d=True)[0].T.copy())
            for name in DEFAULT_SOURCES
        ])
        torch.testing.assert_close(actual_stems, expected_stems, rtol=0, atol=0)
        self.assertFalse(torch.allclose(actual_mix, actual_stems.sum(0)))
        self.assertEqual(originals, {name: (path / f"{name}.wav").read_bytes() for name in originals})
        with self.assertRaisesRegex(ValueError, "left channel mixture/stem residual"):
            self.scan(allow_known_errata=False)

    def test_known_track_observed_right_residual_warns_and_strict_mode_rejects_both(self):
        path = self.track(name="PR - Oh No", split="test", subtype="PCM_16", frames=88200)
        mixture, sr = sf.read(path / "mixture.wav", dtype="float32", always_2d=True)
        mixture[:, 0] += np.float32(0.25)
        # Reproduce the newly observed right RMS levels at two spread probes.
        mixture[:44100, 1] += np.float32(0.031924)
        mixture[44100:, 1] += np.float32(0.021888)
        sf.write(path / "mixture.wav", mixture, sr, subtype="PCM_16")
        originals = {name: (path / f"{name}.wav").read_bytes() for name in ("mixture", *DEFAULT_SOURCES)}
        output = self.root / "manifest.json"
        with warnings.catch_warnings(record=True) as caught:
            records = self.scan(output=output, probe_seconds=0.25)
        messages = [str(item.message) for item in caught]
        self.assertTrue(any("known MUSDB erratum on left channel" in message for message in messages))
        self.assertTrue(any("additional observed right-channel residual" in message for message in messages))
        self.assertTrue(any("official erratum describes the left channel" in message for message in messages))
        self.assertEqual(records[0]["residual_policy"], "warn_known_artistic_mix")
        self.assertEqual(len(records), 1)
        self.assertTrue(output.exists())
        audit = json.loads((self.root / "manifest.audit.json").read_text())
        self.assertFalse(audit["errors"])
        probes = audit["tracks"][0]["probes"]
        self.assertAlmostEqual(probes[0]["residual_by_channel"]["right"]["residual_rms"], 0.031924, delta=5e-5)
        self.assertAlmostEqual(probes[-1]["residual_by_channel"]["right"]["residual_rms"], 0.021888, delta=5e-5)
        for probe in probes:
            diagnostic = probe["residual_by_channel"]["right"]
            self.assertGreater(diagnostic["residual_rms"], diagnostic["failure_floor"])
            self.assertGreaterEqual(diagnostic["failure_floor"], 20 * 2**-15)
        self.assertEqual(audit["known_errata_policy"]["official_erratum_channels"], ["left"])
        self.assertEqual(audit["known_errata_policy"]["additional_observed_residual_channels"], ["right"])
        self.assertEqual(originals, {name: (path / f"{name}.wav").read_bytes() for name in originals})
        with self.assertRaises(ValueError) as error:
            self.scan(allow_known_errata=False, probe_seconds=0.25)
        self.assertIn("left channel mixture/stem residual", str(error.exception))
        self.assertIn("right channel mixture/stem residual", str(error.exception))

    def test_erratum_is_scoped_to_exact_track_name_and_physical_test_split(self):
        for split, name in (("train", "PR - Oh No"), ("test", "Unknown Residual Song")):
            with self.subTest(split=split, name=name):
                path = self.track(name=name, split=split)
                mixture, sr = sf.read(path / "mixture.wav", dtype="float32", always_2d=True)
                mixture[:, 0] += np.float32(0.25)
                sf.write(path / "mixture.wav", mixture, sr, subtype="FLOAT")
                with self.assertRaisesRegex(ValueError, f"{split}/{name}: mixture/stem residual"):
                    self.scan()

    def test_known_erratum_still_rejects_nonfinite_samples_and_length_mismatch(self):
        path = self.track(name="PR - Oh No", split="test")
        bass, sr = sf.read(path / "bass.wav", dtype="float32", always_2d=True)
        bass[0, 0] = np.nan
        sf.write(path / "bass.wav", bass, sr, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "non-finite"):
            self.scan()
        sf.write(path / "bass.wav", np.zeros((2999, 2)), sr, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "bass.wav: 2999 frames"):
            self.scan()

    def test_training_remix_preserves_stereo_and_exact_sum_with_shared_scale(self):
        self.track(seed=1)
        self.track(name="Second Train Song", seed=2)
        records = self.scan()
        dataset = MUSDBDataset(records, segment_seconds=0.02, crops_per_track=4, remix_prob=1.0, gain_db=15)
        mixture, targets = dataset[0]
        self.assertEqual(tuple(mixture.shape), (2, 882))
        self.assertEqual(tuple(targets.shape), (4, 2, 882))
        self.assertTrue(torch.equal(mixture, targets.sum(dim=0)))
        self.assertLessEqual(float(mixture.abs().max()), 1.0)
        self.assertLessEqual(float(targets.abs().max()), 1.0)
        for source in targets:
            self.assertFalse(torch.equal(source[0], source[1]))
            ratio = float(source[1].norm() / source[0].norm())
            self.assertTrue(abs(ratio - 0.37) < 1e-5 or abs(ratio - 1 / 0.37) < 1e-5)
        self.assertTrue(torch.equal(mixture, dataset[0][0]))
        dataset.set_epoch(1)
        self.assertFalse(torch.equal(mixture, dataset[0][0]))
        bad_records = [dict(records[0], subset="val")]
        with self.assertRaisesRegex(ValueError, "Training dataset includes val"):
            MUSDBDataset(bad_records)

    def test_validation_is_deterministic_uses_original_mixture_and_no_augmentation(self):
        path = self.track(name="Actions - One Minute Smile")
        # A small export residual lets this test distinguish the original
        # mixture from the sum reconstructed for augmented training samples.
        mixture, sr = sf.read(path / "mixture.wav", dtype="float32", always_2d=True)
        mixture += np.float32(1e-6)
        sf.write(path / "mixture.wav", mixture, sr, subtype="FLOAT")
        records = self.scan()
        dataset = MUSDBDataset(records, segment_seconds=0.02, training=False, validation_crops=3)
        expected_offsets = (0, (3000 - 882) // 2, 3000 - 882)
        for index, offset in enumerate(expected_offsets):
            actual, _ = dataset[index]
            expected = torch.from_numpy(mixture[offset:offset + 882].T.copy())
            self.assertTrue(torch.equal(actual, expected))
            dataset.set_epoch(200)
            self.assertTrue(torch.equal(actual, dataset[index][0]))
        alternate_seed = MUSDBDataset(records, segment_seconds=0.02, training=False, validation_crops=3, seed=987)
        self.assertTrue(torch.equal(dataset[1][0], alternate_seed[1][0]))

    def test_stale_manifest_is_rejected_and_short_tracks_are_padded_together(self):
        path = self.track(frames=200)
        records = self.scan()
        dataset = MUSDBDataset(records, segment_seconds=0.02, training=False)
        mixture, targets = dataset[0]
        self.assertTrue(torch.equal(mixture[:, 200:], torch.zeros(2, 682)))
        self.assertTrue(torch.equal(targets[..., 200:], torch.zeros(4, 2, 682)))
        sf.write(path / "drums.wav", np.zeros((201, 2)), 44100, subtype="FLOAT")
        with self.assertRaisesRegex(ValueError, "WAV/manifest mismatch"):
            MUSDBDataset(records)


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile tests/test_evaluate.py
import io
import json
import sys
import tempfile
import unittest
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import patch

import numpy as np
import soundfile as sf
import torch

from audio_sep.data import DEFAULT_SOURCES
from audio_sep.evaluate import evaluation_report


class EvaluationTest(unittest.TestCase):
    def score_row(self, name, score, *, flagged=False):
        row = {"track_name": name, "snr_db": [score] * 4, "si_sdr_db": [score + 1] * 4}
        if flagged:
            row.update(known_issue="musdb18hq_pr_oh_no_left_mixture",
                       residual_policy="warn_known_artistic_mix")
        return row

    def test_primary_includes_flagged_track_and_secondary_excludes_only_flags(self):
        rows = [self.score_row("PR - Oh No", 2, flagged=True), self.score_row("Ordinary", 10)]
        report = evaluation_report(rows, {"subset": "test"})
        self.assertEqual(report["mixture_policy"], "original_mixture_wav")
        self.assertEqual(report["summary"]["macro_snr_db"], 6)
        self.assertEqual(report["summary"]["macro_si_sdr_db"], 7)
        self.assertEqual(report["summary_without_flagged_tracks"]["macro_snr_db"], 10)
        self.assertEqual(report["track_counts"], {"all": 2, "without_flagged": 1, "flagged": 1})
        self.assertEqual(report["flagged_tracks"], [{"track_name": "PR - Oh No",
                         "known_issue": "musdb18hq_pr_oh_no_left_mixture",
                         "residual_policy": "warn_known_artistic_mix"}])
        self.assertEqual(report["tracks"], rows)

    def test_all_flagged_tracks_have_no_secondary_score(self):
        rows = [self.score_row("PR - Oh No", 2, flagged=True)]
        report = evaluation_report(rows, {})
        self.assertEqual(report["summary"]["macro_snr_db"], 2)
        self.assertIsNone(report["summary_without_flagged_tracks"])
        self.assertEqual(report["track_counts"], {"all": 1, "without_flagged": 0, "flagged": 1})

    def test_cli_preserves_original_mixture_and_audit_provenance(self):
        from audio_sep import evaluate

        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            records, original_mixtures, targets = [], [], []
            time = np.arange(882, dtype=np.float32) / 44100
            for index, name in enumerate(("PR - Oh No", "Ordinary")):
                directory = root / name
                directory.mkdir()
                stems = []
                for source_index, source in enumerate(DEFAULT_SOURCES):
                    waveform = 0.03 * np.sin(2 * np.pi * (100 + 70 * source_index) * time)
                    stem = np.stack((waveform, 0.8 * waveform), axis=1).astype(np.float32)
                    sf.write(directory / f"{source}.wav", stem, 44100, subtype="FLOAT")
                    stems.append(stem)
                mixture = np.sum(stems, axis=0)
                if index == 0:
                    # Both-channel residual mirrors the policy decision, without
                    # representing this synthetic audio as the real dataset.
                    mixture = mixture + np.array([0.06, 0.03], dtype=np.float32)
                sf.write(directory / "mixture.wav", mixture, 44100, subtype="FLOAT")
                record = {"track_name": name, "subset": "test", "path": str(directory)}
                if index == 0:
                    record.update(known_issue="musdb18hq_pr_oh_no_left_mixture",
                                  residual_policy="warn_known_artistic_mix",
                                  max_probe_residual_rms_by_channel={"left": 0.06, "right": 0.03})
                records.append(record)
                original_mixtures.append(torch.from_numpy(mixture.T.copy()))
                targets.append(torch.from_numpy(np.stack(stems).transpose(0, 2, 1).copy()))

            manifest = root / "manifest.json"
            manifest.write_text(json.dumps(records))
            output = root / "report.json"
            seen_mixtures = []

            def separate(model, mixture, **kwargs):
                seen_mixtures.append(mixture.clone())
                return targets[len(seen_mixtures) - 1] * 0.5

            model = SimpleNamespace(sources=DEFAULT_SOURCES)
            command = ["audio_sep.evaluate", "--manifest", str(manifest), "--checkpoint", "unused.pth",
                       "--output", str(output), "--device", "cpu"]
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                with patch.object(sys, "argv", command), patch.object(evaluate, "load_checkpoint", return_value=(model, {})):
                    with patch.object(evaluate, "overlap_add", side_effect=separate):
                        evaluate.main()

            self.assertEqual(len(seen_mixtures), 2)
            for actual, expected in zip(seen_mixtures, original_mixtures):
                torch.testing.assert_close(actual, expected)
            self.assertFalse(torch.allclose(seen_mixtures[0], targets[0].sum(0)))
            report = json.loads(output.read_text())
            self.assertEqual(report["mixture_policy"], "original_mixture_wav")
            self.assertEqual(report["track_counts"], {"all": 2, "without_flagged": 1, "flagged": 1})
            for key in ("known_issue", "residual_policy", "max_probe_residual_rms_by_channel"):
                self.assertEqual(report["tracks"][0][key], records[0][key])
            self.assertNotIn("known_issue", report["tracks"][1])


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile tests/test_inference.py
import tempfile
import unittest
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from torch import nn

from audio_sep.data import DEFAULT_SOURCES
from audio_sep.inference import load_audio, load_checkpoint, overlap_add, save_stems
from audio_sep.model import DemucsLiteV2


class Splitter(nn.Module):
    sources = DEFAULT_SOURCES
    samplerate = 100

    def forward(self, mix):
        return mix[:, None].repeat(1, 4, 1, 1) / 4


class InferenceTest(unittest.TestCase):
    def test_overlap_boundaries_and_reconstruction(self):
        for length in (1, 7, 99, 513):
            x = torch.randn(2, length)
            y = overlap_add(Splitter(), x, segment_seconds=1)
            self.assertEqual(y.shape, (4, 2, length))
            torch.testing.assert_close(y.sum(0), x)

    def test_float_export_does_not_clip(self):
        with tempfile.TemporaryDirectory() as temp:
            x = torch.full((4, 2, 21), 1.5)
            save_stems(x, temp)
            y, _ = sf.read(Path(temp) / "vocals.wav")
            np.testing.assert_allclose(y, 1.5)

    def test_mono_becomes_stereo_and_bad_audio_rejected(self):
        with tempfile.TemporaryDirectory() as temp:
            path = Path(temp) / "input.wav"
            sf.write(path, np.ones(20), 44100, subtype="FLOAT")
            y = load_audio(path)
            self.assertEqual(tuple(y.shape), (2, 20))
            torch.testing.assert_close(y[0], y[1])
            sf.write(path, np.full(20, np.nan), 44100, subtype="FLOAT")
            with self.assertRaises(ValueError):
                load_audio(path)

    def test_checkpoint_honors_selected_raw_weights(self):
        model = DemucsLiteV2(channels=4, depth=2, lstm_layers=1)
        raw = model.state_dict()
        ema = {k: v + 1 if v.is_floating_point() else v for k, v in raw.items()}
        checkpoint = {"format_version": 2, "model_kind": "lite", "model_config": model.export_config(),
                      "model_state_dict": raw, "ema_state_dict": ema, "selected_weights": "raw"}
        with tempfile.TemporaryDirectory() as temp:
            path = Path(temp) / "best.pth"
            torch.save(checkpoint, path)
            loaded, _ = load_checkpoint(path)
            for key, value in loaded.state_dict().items():
                torch.testing.assert_close(value, raw[key])


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile tests/test_model_losses.py
import unittest

import torch

from audio_sep.losses import multi_resolution_stft_loss, separation_loss
from audio_sep.metrics import metrics_per_source
from audio_sep.model import DemucsLiteV2, count_parameters


class ModelTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        # Tiny CPU kernels suffer badly from oversized machine thread pools.
        torch.set_num_threads(1)

    def tiny(self, **overrides):
        config = dict(channels=4, depth=2, kernel_size=4, stride=2, lstm_layers=1, dconv_depth=1)
        config.update(overrides)
        return DemucsLiteV2(**config)

    def test_shapes_for_odd_short_and_valid_lengths(self):
        model = self.tiny()
        for length in (1, 5, 31, 129):
            with self.subTest(length=length):
                output = model(torch.randn(1, 2, length))
                self.assertEqual(tuple(output.shape), (1, 4, 2, length))
                self.assertTrue(torch.isfinite(output).all())
                self.assertGreaterEqual(model.valid_length(length), length)
        self.assertGreater(count_parameters(model), 0)

    def test_deep_context_and_disabled_lstm_receive_gradient(self):
        model = self.tiny(depth=3, lstm_layers=0)
        output = model(torch.randn(1, 2, 67))
        output.square().mean().backward()
        self.assertTrue(all(p.grad is not None for p in model.parameters()))
        self.assertTrue(all(torch.isfinite(p.grad).all() for p in model.parameters()))

    def test_export_config_preserves_order_and_exact_predictions(self):
        model = self.tiny(sources=("other", "vocals", "drums", "bass")).eval()
        clone = DemucsLiteV2(**model.export_config()).eval()
        clone.load_state_dict(model.state_dict())
        waveform = torch.randn(2, 2, 73)
        self.assertEqual(clone.sources, model.sources)
        torch.testing.assert_close(clone(waveform), model(waveform))

    def test_normalization_restores_mean_once(self):
        model = self.tiny(lstm_layers=0)
        for parameter in model.parameters():
            parameter.data.zero_()
        mixture = torch.full((1, 2, 71), 0.4)
        estimates = model(mixture)
        torch.testing.assert_close(estimates, torch.full_like(estimates, 0.1))
        torch.testing.assert_close(estimates.sum(dim=1), mixture)

    def test_optional_consistency_preserves_each_stereo_channel(self):
        model = self.tiny(mixture_consistency=True)
        mixture = torch.randn(2, 2, 101) * 0.2 + 0.07
        estimate = model(mixture)
        torch.testing.assert_close(estimate.sum(dim=1), mixture, atol=1e-6, rtol=1e-5)


class LossAndMetricTests(unittest.TestCase):
    def test_antiphase_stereo_is_not_discarded_by_spectral_loss(self):
        left = torch.sin(torch.arange(257).float() * 0.19)
        target = torch.stack((left, -left))[None, None]
        estimate = torch.zeros_like(target, requires_grad=True)
        loss = multi_resolution_stft_loss(estimate, target, fft_sizes=(32, 64))
        self.assertGreater(float(loss), 0.5)
        loss.backward()
        self.assertTrue(torch.isfinite(estimate.grad).all())

    def test_silent_targets_have_finite_nonzero_gradient(self):
        estimate = torch.full((1, 4, 2, 23), 0.01, requires_grad=True)
        target = torch.zeros_like(estimate)
        mixture = target.sum(dim=1)
        loss, parts = separation_loss(estimate, target, mixture, fft_sizes=(32, 64))
        self.assertTrue(torch.isfinite(loss))
        self.assertGreater(float(parts["silence"]), 0)
        loss.backward()
        self.assertTrue(torch.isfinite(estimate.grad).all())
        self.assertGreater(float(estimate.grad.abs().sum()), 0)
        metrics = metrics_per_source(estimate.detach(), target)
        self.assertFalse(metrics["active"].any())
        self.assertTrue(torch.isfinite(metrics["si_sdr_db"]).all())
        self.assertGreater(float(metrics["silent_estimate_energy"].mean()), 0)

    def test_exact_silence_has_no_nan_gradients(self):
        estimate = torch.zeros((1, 4, 2, 17), requires_grad=True)
        loss, _ = separation_loss(estimate, torch.zeros_like(estimate), estimate.detach().sum(dim=1), fft_sizes=(32,))
        loss.backward()
        self.assertTrue(torch.isfinite(estimate.grad).all())
        self.assertAlmostEqual(float(loss), 0.0, places=6)

    def test_equal_relative_error_has_equal_weight_for_quiet_stem(self):
        target = torch.ones((1, 2, 2, 17))
        target[:, 1] *= 0.2
        loss, parts = separation_loss(0.9 * target, target, target.sum(dim=1),
                                      spectral_weight=0, si_sdr_weight=0, consistency_weight=0)
        self.assertAlmostEqual(float(parts["source_l1"]), 0.1, places=5)
        self.assertAlmostEqual(float(loss), 0.1, places=5)

    def test_metrics_distinguish_gain_error_from_scale_invariant_score(self):
        torch.manual_seed(10)
        target = torch.randn(2, 4, 2, 257) * 0.1
        metrics = metrics_per_source(0.5 * target, target)
        self.assertTrue(metrics["active"].all())
        self.assertLess(float((metrics["snr_db"] - 6.0206).abs().max()), 0.02)
        self.assertGreater(float(metrics["si_sdr_db"].mean()), 40)

    def test_stereo_swap_is_detected(self):
        torch.manual_seed(5)
        target = torch.randn(1, 1, 2, 257) * 0.1
        good = metrics_per_source(target, target)
        swapped = metrics_per_source(target.flip(dims=(2,)), target)
        self.assertGreater(float(good["snr_db"]), float(swapped["snr_db"]) + 30)
        self.assertGreater(float(good["si_sdr_db"]), float(swapped["si_sdr_db"]) + 30)


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile tests/test_training.py
import json
import io
import copy
from contextlib import redirect_stderr, redirect_stdout
import sys
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

import numpy as np
import soundfile as sf
import torch

from audio_sep.data import DEFAULT_SOURCES, scan_dataset
from audio_sep.inference import load_checkpoint


class GradientStepTest(unittest.TestCase):
    """CPU GradScaler exercises real overflow recovery without claiming CUDA QA."""

    @classmethod
    def setUpClass(cls):
        torch.set_num_threads(1)

    def test_amp_overflow_skips_weights_ema_and_step_then_recovers(self):
        from audio_sep.train import checked_optimizer_step
        model = torch.nn.Linear(2, 1, bias=False)
        with torch.no_grad():
            model.weight.fill_(0.2)
        ema = copy.deepcopy(model).requires_grad_(False)
        with torch.no_grad():
            ema.weight.fill_(0.8)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
        scaler = torch.amp.GradScaler("cpu", init_scale=16.0)
        weights_before, ema_before = model.weight.detach().clone(), ema.weight.detach().clone()
        scaler.scale(model(torch.ones(1, 2)).square().mean()).backward()
        model.weight.grad[0, 0] = float("inf")
        overflow = checked_optimizer_step(model, optimizer, scaler, step=7, ema=ema)
        self.assertFalse(overflow["stepped"])
        self.assertFalse(overflow["gradient_entries_finite"])
        self.assertEqual(overflow["step"], 7)
        self.assertEqual(overflow["scale_before"], 16.0)
        self.assertEqual(overflow["scale_after"], 8.0)
        self.assertIsNone(model.weight.grad)
        self.assertFalse(optimizer.state)
        torch.testing.assert_close(model.weight, weights_before)
        torch.testing.assert_close(ema.weight, ema_before)
        scaler.scale(model(torch.ones(1, 2)).square().mean()).backward()
        healthy = checked_optimizer_step(model, optimizer, scaler, step=overflow["step"], ema=ema)
        self.assertTrue(healthy["stepped"])
        self.assertEqual(healthy["step"], 8)
        self.assertTrue(torch.isfinite(model.weight).all())
        self.assertFalse(torch.equal(model.weight, weights_before))
        self.assertFalse(torch.equal(ema.weight, ema_before))
        # Step=8 uses decay=9/18. A skipped step must not change that schedule.
        torch.testing.assert_close(ema.weight, 0.5 * ema_before + 0.5 * model.weight)

    def test_finite_large_gradients_clip_with_float64_norm(self):
        from audio_sep.train import checked_optimizer_step
        model = torch.nn.Linear(2, 1, bias=False)
        with torch.no_grad():
            model.weight.zero_()
        model.weight.grad = torch.full_like(model.weight, 1e20)
        # The old float32 norm overflows although all gradient entries are finite.
        self.assertTrue(torch.isfinite(model.weight.grad).all())
        self.assertFalse(torch.isfinite(torch.linalg.vector_norm(model.weight.grad)))
        optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
        scaler = torch.amp.GradScaler("cpu", enabled=False)
        result = checked_optimizer_step(model, optimizer, scaler, max_norm=5.0)
        self.assertTrue(result["stepped"])
        self.assertTrue(result["gradient_entries_finite"])
        self.assertAlmostEqual(result["grad_norm"] / 1e20, 2**0.5, places=5)
        self.assertTrue(torch.isfinite(model.weight).all())
        self.assertAlmostEqual(float(torch.linalg.vector_norm(model.weight.detach())), 5.0, places=5)

    def test_fp32_nonfinite_gradients_remain_fatal(self):
        from audio_sep.train import checked_optimizer_step
        for bad_value in (float("inf"), float("nan")):
            with self.subTest(bad_value=bad_value):
                model = torch.nn.Linear(2, 1, bias=False)
                before = model.weight.detach().clone()
                model.weight.grad = torch.full_like(model.weight, bad_value)
                optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
                scaler = torch.amp.GradScaler("cpu", enabled=False)
                with self.assertRaisesRegex(FloatingPointError, "Nonfinite FP32 gradient entries"):
                    checked_optimizer_step(model, optimizer, scaler)
                torch.testing.assert_close(model.weight, before)
                self.assertIsNone(model.weight.grad)

    def test_scaler_restore_allows_explicit_precision_mode_changes(self):
        from audio_sep.train import restore_scaler_state
        amp = torch.amp.GradScaler("cpu", init_scale=16.0)
        state = amp.state_dict()
        fp32 = torch.amp.GradScaler("cpu", enabled=False)
        self.assertEqual(restore_scaler_state(fp32, state), "disabled")
        self.assertEqual(fp32.state_dict(), {})
        fresh_amp = torch.amp.GradScaler("cpu", init_scale=32.0)
        self.assertEqual(restore_scaler_state(fresh_amp, fp32.state_dict()), "fresh")
        self.assertEqual(fresh_amp.get_scale(), 32.0)
        self.assertEqual(restore_scaler_state(fresh_amp, state), "restored")
        self.assertEqual(fresh_amp.get_scale(), 16.0)

    def test_persistent_amp_overflow_stops_after_bounded_consecutive_groups(self):
        from audio_sep import train
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            manifest = TrainingTest().synthetic_manifest(root)
            run = root / "run"
            command = ["audio_sep.train", "--manifest", str(manifest), "--run-dir", str(run),
                       "--channels", "4", "--depth", "2", "--segment", "0.02",
                       "--train-crops", "12", "--val-crops", "2", "--workers", "0",
                       "--device", "cpu", "--epochs", "1", "--accumulate", "1"]
            scales, steps = [], []

            def persistent_overflow(model, optimizer, scaler, *, step, **kwargs):
                scales.append(16.0 / 2**len(scales))
                steps.append(step)
                optimizer.zero_grad(set_to_none=True)
                return {"stepped": False, "step": step, "scale_before": scales[-1],
                        "scale_after": scales[-1] / 2}

            stdout = io.StringIO()
            with redirect_stdout(stdout), redirect_stderr(io.StringIO()):
                with patch.object(sys, "argv", command), patch.object(train, "checked_optimizer_step", persistent_overflow):
                    with self.assertRaisesRegex(FloatingPointError, "8 consecutive nonfinite-gradient groups"):
                        train.main()
            self.assertEqual(len(scales), train.MAX_CONSECUTIVE_AMP_OVERFLOWS)
            self.assertEqual(steps, [0] * train.MAX_CONSECUTIVE_AMP_OVERFLOWS)
            self.assertIn("scale 16 -> 8", stdout.getvalue())
            self.assertFalse((run / "last.pth").exists())
            self.assertFalse((run / "best.pth").exists())


class TrainingTest(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        torch.set_num_threads(1)

    def synthetic_manifest(self, root):
        sample = np.arange(4410) / 44100
        for subset, name in (("train", "Synthetic Train"),
                             ("train", "Actions - One Minute Smile"),
                             ("test", "Synthetic Test")):
            directory = root / "dataset" / subset / name
            directory.mkdir(parents=True)
            stems = []
            for i, source in enumerate(DEFAULT_SOURCES):
                x = (0.02 * np.sin(2 * np.pi * (110 + 80 * i) * sample)).astype("float32")
                stereo = np.stack((x, 0.7 * x), axis=1)
                sf.write(directory / f"{source}.wav", stereo, 44100, subtype="FLOAT")
                stems.append(stereo)
            sf.write(directory / "mixture.wav", np.sum(stems, axis=0), 44100, subtype="FLOAT")
        manifest = root / "manifest.json"
        scan_dataset(root / "dataset", manifest, strict=False)
        return manifest

    def test_latest_best_export_survives_interruption_in_later_epoch(self):
        from audio_sep import train
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            manifest = self.synthetic_manifest(root)
            run = root / "run"
            command = ["audio_sep.train", "--manifest", str(manifest), "--run-dir", str(run),
                       "--channels", "4", "--depth", "2", "--segment", "0.02",
                       "--train-crops", "2", "--val-crops", "2", "--workers", "0",
                       "--device", "cpu", "--epochs", "2"]
            real_validate = train.validate
            calls = 0

            def interrupt_second_epoch(*args, **kwargs):
                nonlocal calls
                calls += 1
                if calls == 3:  # epoch 1 raw + EMA completed; epoch 2 is interrupted.
                    raise RuntimeError("Simulated runtime interruption")
                return real_validate(*args, **kwargs)

            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                with patch.object(sys, "argv", command), patch.object(train, "validate", side_effect=interrupt_second_epoch):
                    with self.assertRaisesRegex(RuntimeError, "Simulated runtime interruption"):
                        train.main()
            last = torch.load(run / "last.pth", weights_only=False)
            best = torch.load(run / "best.pth", weights_only=False)
            model, export = load_checkpoint(run / "best_model.pth")
            self.assertEqual(last["epoch"], 1)
            self.assertEqual(export["epoch"], best["epoch"])
            self.assertEqual(export["best_snr_db"], best["best_snr_db"])
            selected = best["ema_state_dict"] if best["selected_weights"] == "ema" else best["model_state_dict"]
            for key, value in model.state_dict().items():
                torch.testing.assert_close(value, selected[key])
            self.assertNotIn("optimizer_state_dict", export)
            self.assertTrue(torch.isfinite(model(torch.randn(1, 2, 882))).all())

    def test_pretrained_baseline_export_survives_before_first_training_crop(self):
        from audio_sep import train
        from audio_sep.data import MUSDBDataset
        from audio_sep.pretrained import PretrainedSeparator
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            manifest = self.synthetic_manifest(root)
            run = root / "run"
            model = PretrainedSeparator(
                init_args=[["drums", "bass", "other", "vocals"]],
                init_kwargs={"channels": 8, "t_layers": 1, "t_heads": 2, "dconv_depth": 0,
                             "segment": 0.2, "samplerate": 44100},
            )
            command = ["audio_sep.train", "--manifest", str(manifest), "--run-dir", str(run),
                       "--kind", "pretrained", "--segment", "0.02", "--train-crops", "2",
                       "--val-crops", "2", "--workers", "0", "--device", "cpu", "--epochs", "1"]
            real_getitem = MUSDBDataset.__getitem__

            def interrupt_training_crop(dataset, index):
                if dataset.training:
                    raise RuntimeError("Simulated interruption before training")
                return real_getitem(dataset, index)

            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                with patch.object(sys, "argv", command), patch.object(train, "build_model", return_value=model):
                    with patch.object(MUSDBDataset, "__getitem__", interrupt_training_crop):
                        with self.assertRaisesRegex(RuntimeError, "Simulated interruption before training"):
                            train.main()
            loaded, export = load_checkpoint(run / "best_model.pth")
            self.assertEqual(export["epoch"], 0)
            self.assertEqual(export["selected_weights"], "raw")
            self.assertTrue((run / "best.pth").exists())
            self.assertFalse((run / "last.pth").exists())
            for key, value in loaded.state_dict().items():
                torch.testing.assert_close(value, model.state_dict()[key])

    def test_train_resume_export_and_full_track_evaluation(self):
        with tempfile.TemporaryDirectory() as temporary:
            root = Path(temporary)
            sample = np.arange(4410) / 44100
            for subset, name in (("train", "Synthetic Train"),
                                 ("train", "Actions - One Minute Smile"),
                                 ("test", "Synthetic Test")):
                directory = root / "dataset" / subset / name
                directory.mkdir(parents=True)
                stems = []
                for i, source in enumerate(DEFAULT_SOURCES):
                    x = (0.02 * np.sin(2 * np.pi * (110 + 80 * i) * sample)).astype("float32")
                    stereo = np.stack((x, 0.7 * x), axis=1)
                    sf.write(directory / f"{source}.wav", stereo, 44100, subtype="FLOAT")
                    stems.append(stereo)
                sf.write(directory / "mixture.wav", np.sum(stems, axis=0), 44100, subtype="FLOAT")
            manifest = root / "manifest.json"
            scan_dataset(root / "dataset", manifest, strict=False)
            run = root / "run"
            from audio_sep.train import main as train_main
            from audio_sep.evaluate import main as evaluate_main
            # Invoke the CLI parser in-process: avoids platform OpenMP shared-memory
            # limits in subprocess imports and also keeps this CPU smoke test fast.
            base = ["audio_sep.train", "--manifest", str(manifest),
                    "--run-dir", str(run), "--channels", "4", "--depth", "2",
                    "--segment", "0.02", "--train-crops", "3", "--val-crops", "2",
                    "--batch-size", "1", "--accumulate", "2", "--workers", "0", "--device", "cpu"]
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                with patch.object(sys, "argv", [*base, "--epochs", "1"]):
                    train_main()
                # An enabled scaler state has the same schema on CPU/CUDA. This
                # simulates an existing AMP checkpoint for explicit FP32 resume.
                prior = torch.load(run / "last.pth", weights_only=False)
                prior["scaler_state_dict"] = torch.amp.GradScaler("cpu", init_scale=16.0).state_dict()
                prior["training_config"]["amp_enabled"] = True
                prior["training_config"]["precision_mode"] = "cuda_amp"
                torch.save(prior, run / "last.pth")
                with patch.object(sys, "argv", [*base, "--epochs", "2", "--resume", "--no-amp"]):
                    train_main()
            checkpoint = torch.load(run / "last.pth", weights_only=False)
            self.assertEqual(checkpoint["epoch"], 2)
            self.assertEqual(len(checkpoint["history"]), 2)
            self.assertEqual(checkpoint["scaler_state_dict"], {})
            self.assertEqual(checkpoint["training_config"]["precision_mode"], "fp32")
            self.assertEqual(checkpoint["training_config"]["resume_precision_from"], "cuda_amp")
            self.assertTrue(checkpoint["training_config"]["no_amp"])
            self.assertEqual(checkpoint["history"][-1]["amp_skipped_groups"], 0)
            model, exported = load_checkpoint(run / "best_model.pth")
            self.assertEqual(exported["sources"], list(DEFAULT_SOURCES))
            self.assertNotIn("optimizer_state_dict", exported)
            self.assertTrue(torch.isfinite(model(torch.randn(1, 2, 882))).all())
            report = root / "report.json"
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                with patch.object(sys, "argv", ["audio_sep.evaluate", "--manifest", str(manifest),
                                               "--checkpoint", str(run / "best_model.pth"),
                                               "--output", str(report), "--device", "cpu"]):
                    evaluate_main()
            self.assertEqual(len(json.loads(report.read_text())["tracks"]), 1)

    def test_htdemucs_forward_and_offline_checkpoint(self):
        from audio_sep.pretrained import PretrainedSeparator
        args = [["drums", "bass", "other", "vocals"]]
        kwargs = {"channels": 8, "t_layers": 1, "t_heads": 2, "dconv_depth": 0,
                  "segment": 0.2, "samplerate": 44100}
        model = PretrainedSeparator(init_args=args, init_kwargs=kwargs)
        mix = torch.randn(1, 2, 4410) * 0.1
        model.train()
        y = model(mix)
        self.assertEqual(tuple(y.shape), (1, 4, 2, 4410))
        y.square().mean().backward()
        self.assertTrue(all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None))
        self.assertAlmostEqual(float(model.backbone.segment), 0.2)
        with tempfile.TemporaryDirectory() as temporary:
            path = Path(temporary) / "model.pth"
            torch.save({"format_version": 2, "model_kind": "pretrained",
                        "model_config": model.export_config(), "model_state_dict": model.state_dict()}, path)
            loaded, _ = load_checkpoint(path)
            torch.testing.assert_close(loaded(mix), model.eval()(mix))


if __name__ == "__main__":
    unittest.main()


## 3. Kiểm tra model/loss bằng dữ liệu tổng hợp

Các test này kiểm tra shape, gradient, stereo, im lặng, augmentation, checkpoint và nối các đoạn. Chúng **không chứng minh chất lượng tách nhạc**.


In [ ]:
import os, subprocess, sys
# 29 test data/inference/model/loss/evaluation, 5 test optimizer/AMP và 2 test pipeline Lite.
# Không chạy test constructor/checkpoint/baseline HTDemucs.
LITE_TESTS = [
    'test_data', 'test_inference', 'test_model_losses', 'test_evaluate',
    'test_training.GradientStepTest',
    'test_training.TrainingTest.test_latest_best_export_survives_interruption_in_later_epoch',
    'test_training.TrainingTest.test_train_resume_export_and_full_track_evaluation',
]
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = os.pathsep.join((str(PROJECT / 'tests'), str(PROJECT)))
subprocess.check_call([sys.executable, '-m', 'unittest', *LITE_TESTS, '-v'],
                      cwd=PROJECT, env=test_environment)


## 4. Tìm ZIP trong Drive và giải nén trên Colab

Nếu folder đang ở **Shared with me**, chọn **Organize → Add shortcut → MyDrive**. Shortcut chỉ trỏ tới file/folder dùng chung, không sao chép 22 GB và không tính dung lượng file dùng chung vào quota MyDrive của bạn. Nếu thêm shortcut sau khi mount, chạy `drive.mount('/content/drive', force_remount=True)` rồi chạy lại cell này.

Code ưu tiên `ZIP_PATH` bạn đặt hoặc ZIP qua shortcut đã mount. Nếu tìm thấy 0 ZIP, nó đăng nhập Google và tải file dataset đã xác minh **trực tiếp vào ổ tạm `/content` của Colab**, không vào MyDrive và không về laptop. MyDrive chỉ lưu checkpoint. Tải từng chunk 8 MiB, kiểm tra quyền tải, metadata, dung lượng trống và checksum; ZIP hoàn chỉnh đã kiểm tra được dùng lại. Nếu tìm thấy nhiều ZIP, đặt `ZIP_PATH` rõ ràng.

Cả ZIP local lẫn WAV giải nén dùng dung lượng Colab; runtime vẫn cần đủ ổ đĩa. Phần giải nén kiểm tra dung lượng riêng sau khi đọc danh sách trong ZIP. Nếu có thể dùng shortcut đã mount, không cần lưu thêm một bản ZIP vào `/content`.


In [ ]:
import os, shutil, zipfile, hashlib
DATASET_ARCHIVE_ID = '1leTZccqRToe17S7DcDXSRsqovnwNaZb5'
DATASET_ARCHIVE_NAME = 'musdb18hq.zip'
DATASET_ARCHIVE_SIZE = 22656664047  # metadata của file dataset đã xác minh
DOWNLOAD_CHUNK_BYTES = 8 * 1024 * 1024
DISK_RESERVE_BYTES = 3 * 2**30

def complete_zip_verified(path, expected_size, expected_md5=None):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != expected_size:
        return False
    if expected_md5:
        digest = hashlib.md5(usedforsecurity=False)  # checksum Drive, không dùng cho bảo mật
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(DOWNLOAD_CHUNK_BYTES), b''):
                digest.update(chunk)
        if digest.hexdigest().lower() != expected_md5.lower():
            return False
    if not zipfile.is_zipfile(path):
        return False
    if not expected_md5:
        # Nếu Drive không cung cấp MD5, kiểm tra CRC đầy đủ trước khi dùng cache.
        with zipfile.ZipFile(path) as archive:
            if archive.testzip() is not None:
                return False
    return True

def download_dataset_zip():
    # Chỉ chạy nhánh này khi không có ZIP qua Drive mount/đường dẫn explicit.
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    from tqdm.auto import tqdm
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    metadata = service.files().get(
        fileId=DATASET_ARCHIVE_ID, supportsAllDrives=True,
        fields='id,name,size,md5Checksum,mimeType,capabilities(canDownload)',
    ).execute()
    if (metadata.get('id') != DATASET_ARCHIVE_ID
            or metadata.get('name') != DATASET_ARCHIVE_NAME
            or int(metadata.get('size', -1)) != DATASET_ARCHIVE_SIZE):
        raise RuntimeError('Metadata ZIP khác file dataset đã xác minh. Kiểm tra lại file ID/name/size.')
    if not metadata.get('capabilities', {}).get('canDownload', False):
        raise PermissionError('Tài khoản Google hiện tại không có quyền tải dataset. Đăng nhập tài khoản được chia sẻ file.')
    checksum = metadata.get('md5Checksum')
    local_zip = Path('/content') / DATASET_ARCHIVE_NAME
    partial = Path(str(local_zip) + '.part')
    if complete_zip_verified(local_zip, DATASET_ARCHIVE_SIZE, checksum):
        print('Dùng ZIP local đã xác minh:', local_zip)
        return local_zip
    if complete_zip_verified(partial, DATASET_ARCHIVE_SIZE, checksum):
        partial.replace(local_zip)
        print('Đã hoàn tất ZIP local từ file .part đã xác minh:', local_zip)
        return local_zip
    free = shutil.disk_usage('/content').free
    reclaimable_partial = partial.stat().st_size if partial.is_file() else 0
    print(f'ZIP cần {DATASET_ARCHIVE_SIZE/2**30:.1f} GiB trên Colab; còn trống {free/2**30:.1f} GiB')
    if free + reclaimable_partial < DATASET_ARCHIVE_SIZE + DISK_RESERVE_BYTES:
        raise RuntimeError('Thiếu dung lượng Colab để tải ZIP. Dùng shortcut đã mount hoặc runtime có ổ đĩa lớn hơn. Không cần copy ZIP vào MyDrive.')
    request = service.files().get_media(fileId=DATASET_ARCHIVE_ID, supportsAllDrives=True)
    # Stream vào .part, không giữ archive 22 GB trong RAM.
    with partial.open('wb') as handle, tqdm(total=DATASET_ARCHIVE_SIZE, unit='B', unit_scale=True,
                                          desc='Drive → /content') as progress:
        downloader = MediaIoBaseDownload(handle, request, chunksize=DOWNLOAD_CHUNK_BYTES)
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=3)
            progress.update(handle.tell() - progress.n)
        handle.flush()
        os.fsync(handle.fileno())
    if not complete_zip_verified(partial, DATASET_ARCHIVE_SIZE, checksum):
        raise RuntimeError('ZIP tải về không đạt kiểm tra size/checksum/CRC. File .part chưa được dùng; chạy lại cell để tải lại.')
    partial.replace(local_zip)  # thay thế atomic sau khi đã kiểm tra hoàn chỉnh
    print('Đã tải và xác minh ZIP trên Colab:', local_zip)
    return local_zip

def resolve_zip_path(explicit_path=None):
    if explicit_path is not None:
        path = Path(explicit_path)
        if not path.is_file():
            raise FileNotFoundError('ZIP_PATH không tồn tại. Kiểm tra đường dẫn shortcut hoặc force_remount Drive.')
        return path
    matches = list(Path('/content/drive/MyDrive').rglob(DATASET_ARCHIVE_NAME))
    if len(matches) > 1:
        raise RuntimeError(f'Tìm thấy {len(matches)} ZIP. Đặt ZIP_PATH chính xác rồi chạy lại: {matches}')
    if matches:
        return matches[0]
    print('Không tìm thấy ZIP qua Drive mount. Tải bằng tài khoản được chia sẻ vào /content của Colab.')
    return download_dataset_zip()

# Giữ đường dẫn đã đặt ở cell trước; không cần hardcode tên folder MyDrive.
ZIP_PATH = resolve_zip_path(globals().get('ZIP_PATH'))
EXTRACT_ROOT = Path('/content/musdb18hq_data')
EXTRACT_ROOT.mkdir(exist_ok=True)
marker = EXTRACT_ROOT / '.extracted.json'
archive_signature = {'path': str(ZIP_PATH), 'size': ZIP_PATH.stat().st_size,
                     'mtime_ns': ZIP_PATH.stat().st_mtime_ns}
import json
if marker.exists():
    if json.loads(marker.read_text()) != archive_signature:
        raise RuntimeError('Thư mục đã chứa dataset khác. Chọn EXTRACT_ROOT mới.')
    print('Dùng dataset đã giải nén.')
else:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = [m for m in archive.infolist() if m.filename.lower().endswith('.wav')]
        if not members:
            raise RuntimeError('ZIP không có WAV.')
        remaining = sum(m.file_size for m in members
                        if not (EXTRACT_ROOT / m.filename).is_file()
                        or (EXTRACT_ROOT / m.filename).stat().st_size != m.file_size)
        free = shutil.disk_usage('/content').free
        print(f'Cần giải nén còn lại {remaining/2**30:.1f} GiB; còn trống {free/2**30:.1f} GiB')
        if free < remaining + 3 * 2**30:
            raise RuntimeError('Thiếu dung lượng Colab. Dùng runtime có nhiều disk hoặc dataset đã giải nén trên Drive.')
        from tqdm.auto import tqdm
        for member in tqdm(members, desc='Extract WAV'):
            destination = (EXTRACT_ROOT / member.filename).resolve()
            if not destination.is_relative_to(EXTRACT_ROOT.resolve()):
                raise RuntimeError('ZIP chứa đường dẫn không an toàn.')
            if destination.is_file() and destination.stat().st_size == member.file_size:
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_suffix('.wav.extracting')
            with archive.open(member) as src, temporary.open('wb') as dst:
                shutil.copyfileobj(src, dst, 8 * 1024 * 1024)
            temporary.replace(destination)
    marker.write_text(json.dumps(archive_signature))


## 5. Audit và split chính thức

100 bài train gốc được chia **86 train + 14 validation** theo MUSDB; 50 bài test giữ riêng. Kiểm tra 5 WAV/bài, 44.1 kHz stereo, chiều dài và probe finite/mix–sum. Probe không thay thế kiểm tra mọi sample.

Riêng `test/PR - Oh No`, SigSep ghi nhận mixture kênh trái không bằng tổng stems. Dataset của bạn còn có residual kênh phải quan sát được khoảng 0,032/0,022 RMS ở hai probe; phần này được ghi là quan sát thêm, không được gán cho errata kênh phải của SigSep. Scanner mặc định áp dụng `warn_known_artistic_mix` cho đúng track test này: ghi cảnh báo cả hai kênh, giữ nguyên mixture/stems cùng đủ 50 bài test và lưu diagnostics. Ngoại lệ không áp dụng cho track khác, cùng tên trong train, sai metadata/shape hoặc sample không hữu hạn. Manifest ghi `known_issue`, `residual_policy` và residual probe lớn nhất từng kênh. Dùng `allow_known_errata=False` để từ chối residual trên cả hai kênh. [Errata chính thức cho kênh trái](https://sigsep.github.io/datasets/musdb.html#errata)


In [ ]:
from audio_sep.data import scan_dataset
MANIFEST = PROJECT / 'manifest.json'
records = scan_dataset(EXTRACT_ROOT, MANIFEST)
from collections import Counter
print(Counter(r['subset'] for r in records))
shutil.copy2(MANIFEST, SAVE_ROOT / 'manifest.json')
shutil.copy2(PROJECT / 'manifest.audit.json', SAVE_ROOT / 'manifest.audit.json')


## 6. Chọn cấu hình huấn luyện

Batch 1 + accumulation 4 giảm VRAM. AMP dùng GradScaler để bỏ bước gradient lỗi và hạ scale; code ghi số bước bỏ qua, không cập nhật EMA/step khi bỏ bước. Với lỗi nonfinite ở run đang mở, đặt TRAIN_NO_AMP=True để resume bằng FP32. Chỉ đổi chế độ AMP/FP32 được resume cùng run; giữ nguyên manifest, segment, crops, LR và loss. Norm được tính float64 để tránh overflow khi mọi gradient vẫn hữu hạn. T4 có thể cần `TRAIN_SEGMENT=4`, hoặc tắt AMP nếu gặp gradient không hữu hạn. Đổi cấu hình phải dùng run name mới; resume chỉ dùng đúng cấu hình/dataset. Cần nhiều phiên Colab nếu train dài.

HTDemucs_ft dùng pretrained có dữ liệu bổ sung, nên mức điểm của nó không phải phép so sánh thuần kiến trúc với model train từ đầu trên 86 bài.


In [ ]:
DO_TRAIN_LITE = True
DO_FINETUNE_HTDEMUCS = False
DO_TEST = False
RUN_NAME = 'v2_run_01'  # đổi tên cho một thí nghiệm khác
LITE_EPOCHS = 100
FINETUNE_EPOCHS = 20
TRAIN_SEGMENT = 6.0
TRAIN_CROPS = 32
VAL_CROPS = 8
LITE_DIR = SAVE_ROOT / RUN_NAME / 'lite'
HT_DIR = SAVE_ROOT / RUN_NAME / 'htdemucs_finetune'

TRAIN_WORKERS = 2
TRAIN_NO_AMP = False
"""Install a Colab train_run helper that preserves the actual child traceback.

Upload this file to Colab, run ``%run -i /content/colab_train_with_logs.py``,
then run the existing Train cell. Importing/running this helper does not train.
It does not change AMP, crop lengths, epochs, or the extracted dataset.
"""
import collections as _trainlog_collections
import datetime as _trainlog_datetime
import os as _trainlog_os
from pathlib import Path as _TrainLogPath
import shlex as _trainlog_shlex
import shutil as _trainlog_shutil
import signal as _trainlog_signal
import subprocess as _trainlog_subprocess
import sys as _trainlog_sys
import uuid as _trainlog_uuid


class TrainRunError(RuntimeError):
    """Training failure with the real exit code and an intact local log."""

    def __init__(self, message, *, returncode, log_path, command):
        super().__init__(message)
        self.returncode = returncode
        self.log_path = str(log_path)
        self.command = command


def _trainlog_stop_child(child):
    if child.poll() is not None:
        if _trainlog_os.name == "posix":
            try:
                _trainlog_os.killpg(child.pid, _trainlog_signal.SIGKILL)
            except ProcessLookupError:
                pass
        return

    def signal_child(sig):
        try:
            if _trainlog_os.name == "posix":
                _trainlog_os.killpg(child.pid, sig)
            elif sig == _trainlog_signal.SIGTERM:
                child.terminate()
            else:
                child.kill()
        except ProcessLookupError:
            pass

    signal_child(_trainlog_signal.SIGTERM)
    try:
        child.wait(timeout=5)
    except _trainlog_subprocess.TimeoutExpired:
        signal_child(_trainlog_signal.SIGKILL)
        child.wait(timeout=5)
    finally:
        # A DataLoader descendant may outlive the main child after SIGTERM.
        if _trainlog_os.name == "posix":
            signal_child(_trainlog_signal.SIGKILL)


def train_run(kind, directory, epochs, lr, patience):
    """Stream training output, save each attempt, and expose its actual error."""
    if kind not in ("lite", "pretrained"):
        raise ValueError("kind phải là lite hoặc pretrained")
    if "PROJECT" not in globals() or "MANIFEST" not in globals():
        raise RuntimeError("Chạy cell ghi module và Audit trước: thiếu PROJECT hoặc MANIFEST.")
    project = _TrainLogPath(globals()["PROJECT"]).expanduser().resolve()
    if not (project / "audio_sep" / "train.py").is_file():
        raise FileNotFoundError(f"Không tìm thấy {project / 'audio_sep' / 'train.py'}")
    manifest = _TrainLogPath(globals()["MANIFEST"]).expanduser()
    if not manifest.is_absolute():
        manifest = project / manifest
    if not manifest.is_file():
        raise FileNotFoundError(f"Chưa có manifest: {manifest}. Chạy cell Audit trước.")
    directory = _TrainLogPath(directory).expanduser()
    if not directory.is_absolute():
        directory = project / directory
    command = [_trainlog_sys.executable, "-u", "-m", "audio_sep.train", "--manifest", str(manifest),
               "--run-dir", str(directory), "--kind", kind, "--epochs", str(epochs),
               "--batch-size", "1", "--accumulate", "4", "--segment", str(globals().get("TRAIN_SEGMENT", 6.0)),
               "--train-crops", str(globals().get("TRAIN_CROPS", 32)),
               "--val-crops", str(globals().get("VAL_CROPS", 8)),
               "--workers", str(globals().get("TRAIN_WORKERS", 2)), "--lr", str(lr), "--patience", str(patience)]
    if globals().get("TRAIN_NO_AMP", False):
        command.append("--no-amp")
    if (directory / "last.pth").exists():
        command.append("--resume")
    logs = project / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    stamp = _trainlog_datetime.datetime.now(_trainlog_datetime.timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    log_path = logs / f"train_{kind}_{stamp}_{_trainlog_uuid.uuid4().hex[:8]}.log"
    tail = _trainlog_collections.deque(maxlen=50)
    environment = _trainlog_os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    child = None
    returncode = None
    print(f"Python: {_trainlog_sys.version.split()[0]} | executable: {_trainlog_sys.executable}", flush=True)
    print(f"cwd: {project}\nLog: {log_path}\nCommand: {_trainlog_shlex.join(command)}", flush=True)
    try:
        with log_path.open("w", encoding="utf-8", buffering=1) as log:
            log.write(f"cwd: {project}\nCommand: {_trainlog_shlex.join(command)}\n")
            try:
                child = _trainlog_subprocess.Popen(
                    command, cwd=str(project), env=environment,
                    stdout=_trainlog_subprocess.PIPE, stderr=_trainlog_subprocess.STDOUT,
                    text=True, encoding="utf-8", errors="replace", bufsize=1,
                    start_new_session=_trainlog_os.name == "posix",
                )
                for line in child.stdout:
                    tail.append(line.rstrip("\n"))
                    log.write(line)
                    log.flush()
                    print(line, end="", flush=True)
                returncode = child.wait()
                log.write(f"\nChild exit code: {returncode}\n")
            except KeyboardInterrupt:
                log.write("\nInterrupted: stopping training child and its process group.\n")
                if child is not None:
                    _trainlog_stop_child(child)
                raise
            except BaseException:
                if child is not None:
                    _trainlog_stop_child(child)
                raise
            finally:
                if child is not None and child.stdout is not None:
                    child.stdout.close()
    finally:
        # Drive can be full/disconnected; retain the local log in either case.
        try:
            drive_logs = directory / "logs"
            drive_logs.mkdir(parents=True, exist_ok=True)
            _trainlog_shutil.copy2(log_path, drive_logs / log_path.name)
        except OSError as error:
            print(f"Chưa copy được log vào run-dir: {error}. Log local vẫn ở {log_path}", flush=True)
    if returncode:
        details = "\n".join(tail) or "Không có output từ tiến trình con; xem file log."
        raise TrainRunError(f"Train dừng với exit code {returncode}. Log: {log_path}\n"
                            f"--- 50 dòng cuối của tiến trình train ---\n{details}",
                            returncode=returncode, log_path=log_path, command=command)
    print(f"Train hoàn tất. Log: {log_path}", flush=True)
    return log_path


print("Đã cài train_run có log. Chạy tiếp cell Train; helper này chưa bắt đầu train.", flush=True)


## 7. Train DemucsLite V2

Mỗi epoch lưu `last.pth` và `history.csv`; khi validation SNR tăng lưu `best.pth` và xuất ngay `best_model.pth` chỉ chứa trọng số được chọn. Nếu phiên bị ngắt ở epoch sau, bản best đã xuất vẫn nằm trên Drive.


In [ ]:
if DO_TRAIN_LITE:
    train_run('lite', LITE_DIR, LITE_EPOCHS, 3e-4, 15)


## 8. Fine-tune HTDemucs

Dùng một backbone `htdemucs`, không train ensemble `htdemucs_ft`. LR thấp 1e-5. Điểm pretrained trước fine-tune là ứng viên epoch 0 và được xuất ngay thành `best_model.pth`, nên nếu các epoch sau kém hơn hoặc phiên bị ngắt thì file best giữ pretrained. Trọng số từ bước tải chính thức mới được nạp với chế độ tương thích PyTorch 2.6; không nạp checkpoint lạ.


In [ ]:
if DO_FINETUNE_HTDEMUCS:
    train_run('pretrained', HT_DIR, FINETUNE_EPOCHS, 1e-5, 5)


## 9. So sánh toàn bộ 50 bài test

SNR và SI-SDR báo riêng, lấy median các cửa sổ 1 giây có tín hiệu trong từng bài, rồi lấy mean theo bài/stem. Không gọi các số này là BSS Eval SDR trong paper. Test chỉ dùng để báo kết quả sau khi khóa cấu hình.

Thêm checkpoint cũ vào `OLD_CHECKPOINT` để so với bản gốc. Bản gốc split random 85/15 khác split hiện tại: chỉ dùng test cho phép so sánh với nó. Đánh giá ensemble 4 model và nhiều shifts tốn thời gian hơn.


In [ ]:
OLD_CHECKPOINT = None  # Path('/content/drive/MyDrive/.../demucs_lite_best.pth')
REPORT_DIR = SAVE_ROOT / RUN_NAME / 'evaluation'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
jobs = [('htdemucs_ft', ['--official', 'htdemucs_ft'])]
if (LITE_DIR / 'best_model.pth').exists():
    jobs.append(('lite_v2', ['--checkpoint', str(LITE_DIR / 'best_model.pth')]))
if (HT_DIR / 'best_model.pth').exists():
    jobs.append(('htdemucs_finetune', ['--checkpoint', str(HT_DIR / 'best_model.pth')]))
if OLD_CHECKPOINT:
    jobs.append(('lite_original', ['--legacy-checkpoint', str(OLD_CHECKPOINT)]))
if DO_TEST:
    for name, options in jobs:
        subprocess.check_call([sys.executable, '-m', 'audio_sep.evaluate', '--manifest', str(MANIFEST),
                               '--subset', 'test', '--output', str(REPORT_DIR / f'{name}.json'), *options])
    for name, _ in jobs:
        report = json.loads((REPORT_DIR / f'{name}.json').read_text())
        print(name, json.dumps(report['summary'], ensure_ascii=False))


## 10. Nghe thử trên cùng một bài

Float WAV giữ biên độ để đánh giá; phát nghe chỉ normalize cho player. Không normalize từng stem rồi dùng file đã normalize để tính metric. Muốn so nhiều model, đổi `MODE` và cùng `INPUT_AUDIO`.


In [ ]:
INPUT_AUDIO = Path(next(r['path'] for r in records if r['subset']=='test')) / 'mixture.wav'
MODE = 'lite_v2'  # 'htdemucs_ft' | 'lite_v2' | 'htdemucs_finetune'
OUTPUT_AUDIO = SAVE_ROOT / RUN_NAME / 'listening' / MODE
options = ['--official', MODE] if MODE == 'htdemucs_ft' else [
    '--checkpoint', str((LITE_DIR if MODE == 'lite_v2' else HT_DIR) / 'best_model.pth')]
subprocess.check_call([sys.executable, '-m', 'audio_sep.inference', str(INPUT_AUDIO),
                       '--output', str(OUTPUT_AUDIO), *options])
from IPython.display import Audio, display
import soundfile as sf
for source in ('vocals','drums','bass','other'):
    audio, sr = sf.read(OUTPUT_AUDIO / f'{source}.wav', always_2d=True)
    print(source)
    display(Audio(audio[:30*sr].T, rate=sr))


## 11. Export best `.pth` và download

File Drive được lưu kể cả khi trình duyệt đóng. Đừng thay thẳng checkpoint V2 vào API cũ: loader phải dựng đúng architecture/config và source order; xem `audio_sep.inference.load_checkpoint`. Bản HTDemucs export chứa constructor config để load không cần tải pretrained lại, nhưng cần cài thư viện Demucs.


In [ ]:
from audio_sep.train import export_best
for directory in (LITE_DIR, HT_DIR):
    if (directory / 'best.pth').exists():
        export_best(directory)  # cũng dùng được nếu dừng train sau một epoch
        print('Đã xuất:', directory / 'best_model.pth')
DOWNLOAD_CHECKPOINTS = False  # đổi True để tải về máy; file vẫn có sẵn trên Drive
if DOWNLOAD_CHECKPOINTS:
    from google.colab import files
    for directory in (LITE_DIR, HT_DIR):
        if (directory / 'best_model.pth').exists():
            files.download(str(directory / 'best_model.pth'))


Nguồn: [Demucs chính thức](https://github.com/facebookresearch/demucs), [code suy luận](https://github.com/facebookresearch/demucs/blob/main/demucs/apply.py), [MUSDB validation split](https://github.com/sigsep/sigsep-mus-db/blob/master/musdb/configs/mus.yaml), [Hybrid Demucs paper](https://arxiv.org/abs/2111.03600).
